# ResNet-50 + CBAM + Focal Loss + MixUp para FER — experimento reproducible (Frontiers)

Este cuaderno reproduce el experimento del artículo y añade lo que pediría un revisor:

| Sección | Qué hace |
|---|---|
| 1 | Entorno y semillas: versiones, GPU, semilla y tiempo de entrenamiento (Tabla 3) |
| 2 | **Configuración — lo único que hay que tocar** |
| 3–5 | Datos sin fuga: validación separada *antes* de balancear, conteo de originales/generadas, duplicados entre particiones |
| 6–9 | Modelo (ResNet-50 + CBAM tras `layer4`), pérdida focal, MixUp, descongelamiento progresivo, parada temprana, TTA de 3 vistas, exactitud de entrenamiento *sin mezcla* |
| 10 | Entrenamiento del modelo principal |
| 11–12 | Predicciones por imagen, McNemar exacto (con/sin TTA), bootstrap pareado, Wilson por clase, F2, errores tipo I/II |
| 12b | Las mismas estadísticas sobre el **checkpoint original** del artículo (sin reentrenar) |
| 13 | Ablación por componentes y por posición de CBAM, 5 semillas, media ± DT, McNemar + Holm |
| 14 | Grad-CAM de aciertos y errores por clase |
| 15 | Salidas para Overleaf: 8 figuras (PNG + PDF, 300 ppp), tablas LaTeX y `resumen_COMPLETAR.md` |

**Prueba rápida:** `FER_SMOKE=1 jupyter nbconvert --to notebook --execute FER_ResNet50_CBAM_Frontiers.ipynb --output /tmp/smoke.ipynb`
genera datos sintéticos diminutos fuera del repositorio y ejecuta todo (incluida la ablación) en unos minutos.

**Ojo con el autosync de `~/TESIS` a GitHub:** los datos generados y los checkpoints (`.pth`) se guardan en `WORK_DIR`
(`~/fer_frontiers_work`, fuera del repositorio). En la carpeta del artículo sólo quedan figuras, tablas, CSV y JSON.

## 1. Importaciones y entorno

In [1]:
import os, sys, json, time, math, random, platform, shutil, subprocess, hashlib, copy
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

import sklearn
import scipy
from scipy import stats
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             fbeta_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

try:
    import imagehash
    HAVE_IMAGEHASH = True
except ImportError:
    HAVE_IMAGEHASH = False   # se usa un dHash propio equivalente

SMOKE = os.environ.get("FER_SMOKE", "0") == "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Modo SMOKE:", SMOKE, "| dispositivo:", device, "| imagehash:", HAVE_IMAGEHASH)

Modo SMOKE: False | dispositivo: cuda | imagehash: False


## 2. Configuración — **cambia aquí**

* `DATA_DIR`: carpeta con `train/`, `validation/` (o `val/`, opcional) y `test/`, una subcarpeta por clase.
* `batch_size`, `max_epochs`, `mixup_prob`: ya están los valores de la ejecución original (`ResNetAttentionFocalLossMixUp-DataBalanceada.ipynb`): 32, 1000 (con parada temprana, paciencia 15) y 1.0 (MixUp en todos los lotes, α = 0.3).
* `stages`: descongelamiento progresivo con las tasas de aprendizaje del código original. La tasa de CBAM en la etapa 3 (`3e-5`) es el coeficiente que falta en la Tabla 4 del artículo.
* `balance_train=True` (recomendado): parte de las imágenes originales y genera las aumentadas sólo a partir de ellas.
  Si `train/` ya trae las generadas, pon `balance_train=False` y `GENERATED_PREFIX` con el prefijo que las identifica (`aug_`).

In [2]:
NB_DIR = Path.cwd()
CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]

CFG = dict(
    # --- rutas ---
    DATA_DIR="~/TESIS/Caras/Nuevo_dataset",
    train_subdir="train", val_subdir=None, test_subdir="test",   # val_subdir=None -> busca 'validation' o 'val'
    OUT_DIR=str(NB_DIR / "resultados_frontiers"),                 # ligero: figuras, tablas, CSV (se versiona)
    WORK_DIR=str(Path.home() / "fer_frontiers_work"),              # pesado: datos generados y .pth (fuera de git)
    original_checkpoint="~/TESIS/Caras/modelos_guardados/best_resnet50_cbam_focal_mixup_balanceado.pth",

    # --- datos ---
    seed=42,
    balance_train=True, GENERATED_PREFIX="aug_",
    aug_out_size=224,          # tamaño de las imágenes generadas (224 como en el artículo; None = tamaño original)
    val_fraction=0.10,         # sólo si no hay carpeta de validación: se separa ANTES de balancear
    dup_hamming=0,             # distancia de Hamming máxima entre hashes perceptuales para considerar duplicado
    remove_cross_duplicates=False,  # True: quita de val/test las imágenes duplicadas con train antes de entrenar

    # --- entrenamiento (valores de la ejecución original) ---
    img_size=224, batch_size=32, max_epochs=1000, patience=15, min_delta=1e-4,
    mixup_alpha=0.3, mixup_prob=1.0, focal_gamma=2.0,
    weight_decay=1e-4, plateau_factor=0.5, plateau_patience=4,
    dropout1=0.5, dropout2=0.4, cbam_reduction=16, cbam_kernel=7,
    stages=[
        dict(start_epoch=0,  train=["cbam", "head"],                     lr={"cbam": 1e-4, "head": 3e-4}),
        dict(start_epoch=10, train=["layer4", "cbam", "head"],           lr={"layer4": 1e-5, "cbam": 5e-5, "head": 1e-4}),
        dict(start_epoch=25, train=["layer3", "layer4", "cbam", "head"], lr={"layer3": 5e-6, "layer4": 1e-5, "cbam": 3e-5, "head": 8e-5}),
    ],
    pretrained=True, use_amp=True, deterministic=False, num_workers=8,
    clean_train_eval_n=3000,   # nº de imágenes ORIGINALES de train para la exactitud de entrenamiento sin mezcla

    # --- estadística ---
    n_bootstrap=10000,

    # --- ablación (45 entrenamientos: 9 variantes x 5 semillas) ---
    RUN_ABLATION=False,
    ablation_seeds=[42, 43, 44, 45, 46],
    ablation_variants=["completo", "base", "sin_cbam", "sin_focal", "sin_mixup",
                       "cbam_layer2", "cbam_layer3", "cbam_layer3_4", "cbam_layer2_3_4"],
    # Versión reducida si hay pocos recursos:
    # ablation_variants=["completo", "base", "sin_cbam", "sin_focal", "sin_mixup"], ablation_seeds=[42, 43, 44]

    retrain=False,   # False: si una ejecución ya terminó (meta.json), se reutiliza en lugar de reentrenar
)

VARIANTS = {
    "completo":        dict(cbam=("layer4",), loss="focal", mixup=True),
    "base":            dict(cbam=(),          loss="ce",    mixup=False),
    "sin_cbam":        dict(cbam=(),          loss="focal", mixup=True),
    "sin_focal":       dict(cbam=("layer4",), loss="ce",    mixup=True),
    "sin_mixup":       dict(cbam=("layer4",), loss="focal", mixup=False),
    "cbam_layer2":     dict(cbam=("layer2",), loss="focal", mixup=True),
    "cbam_layer3":     dict(cbam=("layer3",), loss="focal", mixup=True),
    "cbam_layer3_4":   dict(cbam=("layer3", "layer4"), loss="focal", mixup=True),
    "cbam_layer2_3_4": dict(cbam=("layer2", "layer3", "layer4"), loss="focal", mixup=True),
}

### 2b. Modo SMOKE (`FER_SMOKE=1`): datos sintéticos diminutos, todo fuera del repositorio

In [3]:
def make_synthetic_dataset(root, seed=0):
    """Imágenes 48x48 en gris con un patrón por clase, desbalanceadas, sin carpeta de validación
    (así se prueba la separación de validación antes del balanceo) y con un duplicado train->test."""
    rng = np.random.RandomState(seed)
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    n_train = dict(angry=16, disgust=5, fear=16, happy=24, neutral=16, sad=16, surprise=12)
    n_test = dict(angry=6, disgust=3, fear=6, happy=8, neutral=6, sad=6, surprise=5)
    for split, counts in [("train", n_train), ("test", n_test)]:
        for k, c in enumerate(CLASSES):
            d = root / split / c
            d.mkdir(parents=True, exist_ok=True)
            for i in range(counts[c]):
                a = rng.normal(110, 25, (48, 48))
                a[4 + 6 * k: 9 + 6 * k, :] += 90          # banda horizontal según la clase
                Image.fromarray(np.clip(a, 0, 255).astype(np.uint8), "L").save(d / f"{split}_{c}_{i}.jpg")
    src = root / "train" / "happy" / "train_happy_0.jpg"
    shutil.copy2(src, root / "test" / "happy" / "dup_of_train_happy_0.jpg")
    return root

if SMOKE:
    smoke_root = Path(os.environ.get("FER_SMOKE_DIR", str(Path.home() / "fer_frontiers_smoke")))
    data_root = make_synthetic_dataset(smoke_root / "data")
    CFG.update(
        DATA_DIR=str(data_root), OUT_DIR=str(smoke_root / "resultados"), WORK_DIR=str(smoke_root / "work"),
        original_checkpoint=None, img_size=64, aug_out_size=64, batch_size=8, max_epochs=4, patience=3,
        stages=[dict(start_epoch=0, train=["cbam", "head"], lr={"cbam": 1e-3, "head": 3e-3}),
                dict(start_epoch=1, train=["layer4", "cbam", "head"], lr={"layer4": 1e-4, "cbam": 5e-4, "head": 1e-3}),
                dict(start_epoch=2, train=["layer3", "layer4", "cbam", "head"],
                     lr={"layer3": 5e-5, "layer4": 1e-4, "cbam": 3e-4, "head": 8e-4})],
        pretrained=False, num_workers=0, clean_train_eval_n=40, n_bootstrap=200,
        RUN_ABLATION=True, ablation_seeds=[0, 1], retrain=True,
    )
    print("SMOKE: datos sintéticos en", data_root)

OUT = Path(CFG["OUT_DIR"]); WORK = Path(CFG["WORK_DIR"])
FIG_DIR, TAB_DIR, RUNS_DIR = OUT / "figuras", OUT / "tablas", WORK / "runs"
for d in (OUT, WORK, FIG_DIR, TAB_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)
NUM_CLASSES = len(CLASSES)
AMP = CFG["use_amp"] and device.type == "cuda"
print("Resultados ->", OUT, "\nPesado     ->", WORK)

Resultados -> resultados_frontiers 
Pesado     -> ~/fer_frontiers_work


### 2c. Semillas y registro del entorno (Tabla 3)

In [4]:
def set_seed(seed, deterministic=CFG["deterministic"]):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def gpu_driver():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                       text=True).strip().splitlines()[0]
    except Exception:
        return "n/a"

ENV = OrderedDict(
    python=platform.python_version(), os=platform.platform(), torch=torch.__version__,
    torchvision=torchvision.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
    numpy=np.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__, pandas=pd.__version__,
    matplotlib=matplotlib.__version__,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    gpu_mem_gb=round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1) if torch.cuda.is_available() else 0,
    driver=gpu_driver(), cpu_count=os.cpu_count(), seed=CFG["seed"], amp=AMP,
    cudnn_deterministic=CFG["deterministic"], smoke=SMOKE,
)
set_seed(CFG["seed"])
json.dump(dict(env=ENV, cfg=CFG), open(OUT / "entorno_y_config.json", "w"), indent=2, default=str)
pd.Series(ENV)

python                                                      3.8.10
os                     Linux-4.19.0-27-amd64-x86_64-with-glibc2.10
torch                                                  2.4.1+cu121
torchvision                                           0.19.1+cu121
cuda                                                          12.1
cudnn                                                        90100
numpy                                                       1.23.5
sklearn                                                      1.3.2
scipy                                                       1.10.1
pandas                                                       2.0.3
matplotlib                                                   3.7.5
gpu                                        NVIDIA GeForce RTX 3090
gpu_mem_gb                                                    23.6
driver                                                  570.153.02
cpu_count                                                     

## 3. Datos sin fuga

1. Se listan las imágenes **originales** de `train/`.
2. Si no hay carpeta de validación, se separa una validación estratificada **antes** de balancear.
3. El balanceo genera imágenes aumentadas **sólo** a partir de las originales de entrenamiento y se guardan en `WORK_DIR`.
4. `test/` se usa con su distribución natural y nunca se balancea.

In [5]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def list_split(folder):
    rows = []
    for k, c in enumerate(CLASSES):
        d = Path(folder) / c
        if d.is_dir():
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(IMG_EXT):
                    rows.append((str(d / f), k))
    return rows

def is_generated(path):
    return Path(path).name.startswith(CFG["GENERATED_PREFIX"])

def source_name(path):
    """aug_<n>_<original> -> <original> (para no usar generadas cuyo origen esté en validación)."""
    n = Path(path).name
    parts = n.split("_", 2)
    return parts[2] if n.startswith(CFG["GENERATED_PREFIX"]) and len(parts) == 3 else n

DATA = Path(CFG["DATA_DIR"])
val_dir = None
for cand in ([CFG["val_subdir"]] if CFG["val_subdir"] else ["validation", "val"]):
    if (DATA / cand).is_dir():
        val_dir = DATA / cand
        break

train_all = list_split(DATA / CFG["train_subdir"])
test_samples = list_split(DATA / CFG["test_subdir"])
train_orig = [s for s in train_all if not is_generated(s[0])]
train_gen_given = [s for s in train_all if is_generated(s[0])]
if CFG["balance_train"] and train_gen_given:
    print(f"AVISO: train/ trae {len(train_gen_given)} imágenes con prefijo '{CFG['GENERATED_PREFIX']}'; "
          "se ignoran y se regenera el balanceo desde las originales.")

if val_dir is not None:
    val_samples = list_split(val_dir)
    VAL_SOURCE = f"carpeta {val_dir.name}/"
else:
    idx = np.arange(len(train_orig))
    tr_idx, va_idx = train_test_split(idx, test_size=CFG["val_fraction"], random_state=CFG["seed"],
                                      stratify=[s[1] for s in train_orig])
    val_samples = [train_orig[i] for i in sorted(va_idx)]
    train_orig = [train_orig[i] for i in sorted(tr_idx)]
    VAL_SOURCE = f"{CFG['val_fraction']:.0%} estratificado de train (antes de balancear)"

print("Validación:", VAL_SOURCE)
print("Originales train:", len(train_orig), "| val:", len(val_samples), "| test:", len(test_samples))

Validación: carpeta validation/
Originales train: 28704 | val: 3588 | test: 3595


## 4. Balanceo de entrenamiento (originales + generadas)

In [6]:
augment_transform = lambda size: transforms.Compose([   # mismo aumento que en el cuaderno original
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(size=size, scale=(0.80, 1.0)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
])

def build_balanced_train(train_orig, seed, tag=""):
    out_root = WORK / f"train_balanced{tag}_seed{seed}_size{CFG['aug_out_size']}"
    manifest = out_root / "manifest.csv"
    if manifest.exists():
        m = pd.read_csv(manifest)
        if set(m.source) <= set(p for p, _ in train_orig) and not SMOKE:
            print("Reutilizando balanceo existente:", out_root)
            return [(r.path, int(r.label)) for r in m.itertuples()]
    if out_root.exists():
        shutil.rmtree(out_root)
    random.seed(seed); torch.manual_seed(seed)
    by_class = {k: [p for p, l in train_orig if l == k] for k in range(NUM_CLASSES)}
    target = max(len(v) for v in by_class.values())
    rows = []
    for k, paths in by_class.items():
        d = out_root / CLASSES[k]; d.mkdir(parents=True, exist_ok=True)
        for i in range(target - len(paths)):
            src = random.choice(paths)
            im = Image.open(src).convert("RGB")
            size = CFG["aug_out_size"] or im.size[0]
            aug = augment_transform(size)(im).convert("L")
            dst = d / f"{CFG['GENERATED_PREFIX']}{i}_{Path(src).stem}.jpg"
            aug.save(dst, quality=95)
            rows.append((str(dst), k, src))
    pd.DataFrame(rows, columns=["path", "label", "source"]).to_csv(manifest, index=False)
    print(f"Generadas {len(rows)} imágenes en {out_root}")
    return [(p, l) for p, l, _ in rows]

if CFG["balance_train"]:
    train_gen = build_balanced_train(train_orig, CFG["seed"])
else:
    val_names = set(Path(p).name for p, _ in val_samples)
    train_gen = [s for s in train_gen_given if source_name(s[0]) not in val_names]
    dropped = len(train_gen_given) - len(train_gen)
    print(f"Generadas ya existentes: {len(train_gen)} (descartadas {dropped} cuyo origen está en validación)")

train_samples = train_orig + train_gen

def counts(samples):
    return np.bincount([l for _, l in samples], minlength=NUM_CLASSES)

data_table = pd.DataFrame({
    "Class": CLASSES,
    "Train original": counts(train_orig), "Train generated": counts(train_gen),
    "Train total": counts(train_samples), "Validation": counts(val_samples), "Test": counts(test_samples),
})
data_table.loc[len(data_table)] = ["Total"] + data_table.iloc[:, 1:].sum().tolist()
data_table["% generated"] = (100 * data_table["Train generated"] / data_table["Train total"]).round(1)
data_table

Reutilizando balanceo existente: WORK_DIR/train_balanced_seed42_size224


,Class,Train original,Train generated,Train total,Validation,Test,% generated
0,angry,3962,3229,7191,495,496,44.9
1,disgust,436,6755,7191,55,56,93.9
2,fear,4096,3095,7191,512,513,43.0
3,happy,7191,0,7191,898,900,0.0
4,neutral,4957,2234,7191,620,621,31.1
5,sad,4861,2330,7191,608,608,32.4
6,surprise,3201,3990,7191,400,401,55.5
7,Total,28704,21633,50337,3588,3595,43.0


## 5. Duplicados entre particiones

Hash perceptual (pHash con `imagehash` si está instalado; si no, dHash propio de 64 bits) y MD5 para copias exactas,
sobre las imágenes **originales** de train, validación y test. Se informa del número de pares y se calculan después las
métricas de test **sin** las imágenes que tienen duplicado en train/val.

In [7]:
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def perceptual_hash(path):
    im = Image.open(path).convert("L")
    if HAVE_IMAGEHASH:
        return int(str(imagehash.phash(im)), 16)
    a = np.asarray(im.resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

def hash_table(samples, name):
    cache = WORK / f"hashes_{name}_{'p' if HAVE_IMAGEHASH else 'd'}.csv"
    if cache.exists() and not SMOKE:
        h = pd.read_csv(cache, dtype={"md5": str})
        if list(h.path) == [p for p, _ in samples]:
            h["phash"] = h.phash.astype(np.uint64)
            return h
    h = pd.DataFrame({"path": [p for p, _ in samples], "label": [l for _, l in samples]})
    h["phash"] = np.array([perceptual_hash(p) for p in h.path], dtype=np.uint64)
    h["md5"] = [md5(p) for p in h.path]
    h.to_csv(cache, index=False)
    return h

def cross_pairs(ha, hb, thr, chunk=128):
    a = ha.phash.values.astype(np.uint64); b = hb.phash.values.astype(np.uint64)
    out = []
    for s in range(0, len(a), chunk):
        x = np.bitwise_xor(a[s:s + chunk, None], b[None, :])
        d = POPCOUNT[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1)
        ii, jj = np.nonzero(d <= thr)
        out += [(s + i, j, int(d[i, j])) for i, j in zip(ii, jj)]
    return out

t0 = time.time()
H = {"train": hash_table(train_orig, "train"), "val": hash_table(val_samples, "val"), "test": hash_table(test_samples, "test")}
dup_rows = []
for A, B in [("train", "val"), ("train", "test"), ("val", "test")]:
    for i, j, d in cross_pairs(H[A], H[B], CFG["dup_hamming"]):
        ra, rb = H[A].iloc[i], H[B].iloc[j]
        dup_rows.append(dict(split_a=A, split_b=B, path_a=ra.path, path_b=rb.path, label_a=CLASSES[ra.label],
                             label_b=CLASSES[rb.label], hamming=d, exact_md5=ra.md5 == rb.md5))
dups = pd.DataFrame(dup_rows, columns=["split_a", "split_b", "path_a", "path_b", "label_a", "label_b", "hamming", "exact_md5"])
dups.to_csv(OUT / "duplicados_entre_particiones.csv", index=False)

dup_summary = (dups.assign(same_label=dups.label_a == dups.label_b)
               .groupby(["split_a", "split_b"])
               .agg(pairs=("hamming", "size"), exact_md5=("exact_md5", "sum"), same_label=("same_label", "sum"))
               .reset_index()) if len(dups) else pd.DataFrame(columns=["split_a", "split_b", "pairs", "exact_md5", "same_label"])
dup_summary["unique_in_b"] = [dups[(dups.split_a == r.split_a) & (dups.split_b == r.split_b)].path_b.nunique()
                              for r in dup_summary.itertuples()]
test_dup_paths = set(dups[dups.split_b == "test"].path_b)
val_dup_paths = set(dups[(dups.split_a == "train") & (dups.split_b == "val")].path_b)
print(f"Hashes en {time.time() - t0:.0f} s. Imágenes de test con duplicado en train/val: {len(test_dup_paths)}")

if CFG["remove_cross_duplicates"]:
    test_samples = [s for s in test_samples if s[0] not in test_dup_paths]
    val_samples = [s for s in val_samples if s[0] not in val_dup_paths]
    print("Duplicados eliminados de val/test ->", len(val_samples), len(test_samples))
dup_summary

Hashes en 8 s. Imágenes de test con duplicado en train/val: 312


,split_a,split_b,pairs,exact_md5,same_label,unique_in_b
0,train,test,383,347,370,274
1,train,val,442,403,418,310
2,val,test,63,56,61,60


## 6. Transformaciones, datasets y TTA de 3 vistas

In [8]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG["img_size"]
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((int(S * 256 / 224), int(S * 256 / 224))),
    transforms.RandomResizedCrop(S, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), transforms.Resize((S, S)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
tta_crop_transform = transforms.Compose([   # vista 3: redimensionar a 236 y recorte central 224
    transforms.Grayscale(num_output_channels=3), transforms.Resize((int(S * 236 / 224), int(S * 236 / 224))),
    transforms.CenterCrop(S), transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class ListDataset(Dataset):
    def __init__(self, samples, transform, tta=False):
        self.samples, self.transform, self.tta = samples, transform, tta
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        im = Image.open(p).convert("RGB")
        if self.tta:   # vistas: original, volteo horizontal, recorte central
            v0 = eval_transform(im)
            x = torch.stack([v0, torch.flip(v0, dims=[2]), tta_crop_transform(im)])
        else:
            x = self.transform(im)
        return x, y, i

def make_loader(samples, transform, shuffle, seed=0, tta=False, drop_last=False):
    g = torch.Generator(); g.manual_seed(seed)
    nw = CFG["num_workers"]
    return DataLoader(ListDataset(samples, transform, tta), batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=nw, pin_memory=device.type == "cuda", drop_last=drop_last,
                      worker_init_fn=seed_worker, generator=g, persistent_workers=False)

rng = np.random.RandomState(CFG["seed"])
clean_idx = rng.choice(len(train_orig), size=min(CFG["clean_train_eval_n"], len(train_orig)), replace=False)
train_clean_samples = [train_orig[i] for i in sorted(clean_idx)]
print("train:", len(train_samples), "| train limpio (originales, sin mezcla):", len(train_clean_samples),
      "| val:", len(val_samples), "| test:", len(test_samples))

train: 50337 | train limpio (originales, sin mezcla): 3000 | val: 3588 | test: 3595


## 7. Modelo: ResNet-50 + CBAM

CBAM con reducción 16 y núcleo espacial 7 tras `layer4` (configuración del artículo). Para la ablación también puede
colocarse tras `layer2` y/o `layer3`. Los nombres de los módulos son los del cuaderno original, así que su checkpoint carga sin cambios.

In [9]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool, self.max_pool = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False), nn.ReLU(inplace=True),
                                nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.conv(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.channel_attention(x)
        return x * self.spatial_attention(x)

class ResNet50CBAM(nn.Module):
    def __init__(self, num_classes=7, cbam_positions=("layer4",), pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = models.resnet50(weights=weights)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        mk = lambda c: CBAM(c, CFG["cbam_reduction"], CFG["cbam_kernel"])
        self.cbam_layer2 = mk(512) if "layer2" in cbam_positions else None
        self.cbam_layer3 = mk(1024) if "layer3" in cbam_positions else None
        self.cbam = mk(2048) if "layer4" in cbam_positions else None
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.LayerNorm(2048), nn.Dropout(CFG["dropout1"]), nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.LayerNorm(512), nn.Dropout(CFG["dropout2"]), nn.Linear(512, num_classes))

    def cbam_modules(self):
        return [m for m in (self.cbam_layer2, self.cbam_layer3, self.cbam) if m is not None]

    def gradcam_layer(self):
        return self.cbam if self.cbam is not None else self.layer4

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer2(self.layer1(x))
        if self.cbam_layer2 is not None: x = self.cbam_layer2(x)
        x = self.layer3(x)
        if self.cbam_layer3 is not None: x = self.cbam_layer3(x)
        x = self.layer4(x)
        if self.cbam is not None: x = self.cbam(x)
        return self.classifier(self.pool(x))

def group_params(model, name):
    if name == "cbam":
        return [p for m in model.cbam_modules() for p in m.parameters()]
    if name == "head":
        return list(model.classifier.parameters())
    return list(getattr(model, name).parameters())

def n_params(params):
    return sum(p.numel() for p in params)

_m = ResNet50CBAM(NUM_CLASSES, ("layer4",), pretrained=False)
PARAMS = dict(cbam=n_params(group_params(_m, "cbam")), head=n_params(group_params(_m, "head")),
              backbone=n_params(_m.parameters()) - n_params(group_params(_m, "cbam")) - n_params(group_params(_m, "head")),
              total=n_params(_m.parameters()))
del _m
print(PARAMS)
assert PARAMS["cbam"] == 524386 and PARAMS["head"] == 1057799, "Los parámetros no coinciden con el artículo"
print("OK: CBAM 524 386 y cabeza 1 057 799 parámetros, como en el artículo.")

{'cbam': 524386, 'head': 1057799, 'backbone': 23508032, 'total': 25090217}
OK: CBAM 524 386 y cabeza 1 057 799 parámetros, como en el artículo.


## 8. Pérdida focal y MixUp

In [10]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def make_criterion(kind):
    c = counts(train_samples).astype(float)
    w = 1.0 / c; w = w / w.sum() * NUM_CLASSES      # con train balanceado todos los pesos valen 1
    w = torch.tensor(w, dtype=torch.float32, device=device)
    return FocalLoss(w, CFG["focal_gamma"]) if kind == "focal" else nn.CrossEntropyLoss(weight=w)

def mixup(x, y, alpha):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

## 9. Entrenamiento y evaluación

* Descongelamiento progresivo según `CFG["stages"]`; en cada cambio de etapa se rehacen AdamW y `ReduceLROnPlateau` (como en el original).
* Parada temprana sobre F1 macro de validación.
* Por época se registran: pérdida y exactitud de entrenamiento **con MixUp** (la del original), exactitud de entrenamiento
  **sin mezcla** sobre originales en modo evaluación (la comparable con validación), métricas de validación, tasas de aprendizaje y tiempo.
* Cada ejecución se guarda en `WORK_DIR/runs/<variante>_s<semilla>/`; si ya terminó, se reutiliza.

In [11]:
import fcntl

def autocast():
    return torch.autocast(device_type=device.type, enabled=AMP)

def apply_stage(model, stage):
    for p in model.parameters():
        p.requires_grad = False
    groups = []
    for name in stage["train"]:
        ps = group_params(model, name)
        if not ps:
            continue                      # p. ej. "cbam" en las variantes sin CBAM
        for p in ps:
            p.requires_grad = True
        groups.append({"params": ps, "lr": stage["lr"][name], "name": name})
    opt = torch.optim.AdamW(groups, weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=CFG["plateau_factor"],
                                                     patience=CFG["plateau_patience"])
    return opt, sch

def train_one_epoch(model, loader, criterion, opt, scaler, use_mixup):
    model.train()
    tot, n, correct = 0.0, 0, 0
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast():
            if use_mixup and np.random.rand() < CFG["mixup_prob"]:
                xm, ya, yb, lam = mixup(x, y, CFG["mixup_alpha"])
                out = model(xm)
                loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
            else:
                out = model(x)
                loss = criterion(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * x.size(0); n += x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    return tot / n, correct / n

@torch.no_grad()
def predict(model, loader, criterion=None, tta=False):
    """Devuelve probabilidades sin TTA y (si tta) con TTA de 3 vistas, más la pérdida media sin TTA."""
    model.eval()
    P0, PT, tot, n = [], [], 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast():
            if tta:
                b, v = x.shape[:2]
                logits = model(x.flatten(0, 1)).float().view(b, v, -1)
                probs = logits.softmax(-1)
                P0.append(probs[:, 0].cpu()); PT.append(probs.mean(1).cpu())
                l0 = logits[:, 0]
            else:
                l0 = model(x).float()
                P0.append(l0.softmax(-1).cpu())
            if criterion is not None:
                tot += criterion(l0, y).item() * y.size(0); n += y.size(0)
    P0 = torch.cat(P0).numpy()
    return P0, (torch.cat(PT).numpy() if tta else None), (tot / n if n else float("nan"))

def labels_of(samples):
    return np.array([l for _, l in samples])

def train_run(variant, seed, verbose=True):
    run_dir = RUNS_DIR / f"{variant}_s{seed}"
    meta_f = run_dir / "meta.json"
    if meta_f.exists() and not CFG["retrain"]:
        print(f"[{variant} s{seed}] ya entrenado: se reutiliza {run_dir}")
        return json.load(open(meta_f))
    run_dir.mkdir(parents=True, exist_ok=True)
    lock_f = open(run_dir / ".lock", "w")   # un solo proceso por corrida: evita que otro sobrescriba best.pth
    try:
        fcntl.flock(lock_f, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError(f"{run_dir} lo está entrenando otro proceso")
    if meta_f.exists() and not CFG["retrain"]:
        return json.load(open(meta_f))
    spec = VARIANTS[variant]
    set_seed(seed)
    model = ResNet50CBAM(NUM_CLASSES, spec["cbam"], CFG["pretrained"]).to(device)
    criterion = make_criterion(spec["loss"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    train_loader = make_loader(train_samples, train_transform, True, seed, drop_last=True)
    val_loader = make_loader(val_samples, eval_transform, False)
    clean_loader = make_loader(train_clean_samples, eval_transform, False)
    y_val, y_clean = labels_of(val_samples), labels_of(train_clean_samples)

    hist, best, bad, best_epoch = [], -1.0, 0, -1
    stage_starts = {s["start_epoch"]: s for s in CFG["stages"]}
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    for epoch in range(CFG["max_epochs"]):
        if epoch in stage_starts:
            opt, sch = apply_stage(model, stage_starts[epoch])
            if verbose:
                print(f"  época {epoch + 1}: etapa {stage_starts[epoch]['train']}")
        t0 = time.time()
        tr_loss, tr_acc_mix = train_one_epoch(model, train_loader, criterion, opt, scaler, spec["mixup"])
        pc, _, _ = predict(model, clean_loader)
        pv, _, va_loss = predict(model, val_loader, criterion)
        va_pred = pv.argmax(1)
        va_f1 = f1_score(y_val, va_pred, average="macro", zero_division=0)
        sch.step(va_f1)
        row = dict(epoch=epoch + 1, train_loss=tr_loss, train_acc_mixup=tr_acc_mix,
                   train_acc_clean=float((pc.argmax(1) == y_clean).mean()), val_loss=va_loss,
                   val_acc=accuracy_score(y_val, va_pred), val_f1_macro=va_f1,
                   lrs={g["name"]: g["lr"] for g in opt.param_groups}, epoch_time_s=time.time() - t0)
        hist.append(row)
        improved = va_f1 > best + CFG["min_delta"]
        if improved:
            best, bad, best_epoch = va_f1, 0, epoch + 1
            torch.save(model.state_dict(), run_dir / "best.pth")
        else:
            bad += 1
        if verbose:
            print(f"  [{epoch + 1:3d}] loss {tr_loss:.4f}/{va_loss:.4f} | acc mix {tr_acc_mix:.3f} limpio "
                  f"{row['train_acc_clean']:.3f} val {row['val_acc']:.3f} | F1 val {va_f1:.4f}"
                  f"{' *' if improved else ''} | {row['epoch_time_s']:.0f}s")
        pd.DataFrame(hist).to_json(run_dir / "history.json", orient="records", indent=1)
        if bad >= CFG["patience"]:
            break
    train_time = time.time() - t_start

    model.load_state_dict(torch.load(run_dir / "best.pth", map_location=device, weights_only=True))
    t0 = time.time()
    p0, pt, _ = predict(model, make_loader(test_samples, None, False, tta=True), tta=True)
    infer_time = time.time() - t0
    save_test_predictions(run_dir / "preds_test.csv", p0, pt)
    meta = dict(variant=variant, seed=seed, spec={k: list(v) if isinstance(v, tuple) else v for k, v in spec.items()},
                epochs_run=len(hist), best_epoch=best_epoch, best_val_f1=best, train_time_s=train_time,
                sec_per_epoch=train_time / len(hist), test_infer_time_tta_s=infer_time,
                peak_gpu_mem_gb=(torch.cuda.max_memory_allocated() / 1024 ** 3) if torch.cuda.is_available() else 0,
                trainable_params_final=n_params([p for p in model.parameters() if p.requires_grad]),
                run_dir=str(run_dir))
    json.dump(meta, open(meta_f, "w"), indent=2)
    lock_f.close()
    del model; torch.cuda.empty_cache()
    return meta

def save_test_predictions(path, p0, pt):
    df = pd.DataFrame({"path": [p for p, _ in test_samples], "label": labels_of(test_samples),
                       "pred": p0.argmax(1), "pred_tta": pt.argmax(1)})
    for k, c in enumerate(CLASSES):
        df[f"p_{c}"] = p0[:, k]
    for k, c in enumerate(CLASSES):
        df[f"ptta_{c}"] = pt[:, k]
    df.to_csv(path, index=False)

def load_test_predictions(run_dir):
    df = pd.read_csv(Path(run_dir) / "preds_test.csv")
    assert list(df.path) == [p for p, _ in test_samples], "El test cambió desde que se guardaron las predicciones"
    return (df.label.values, df[[f"p_{c}" for c in CLASSES]].values, df[[f"ptta_{c}" for c in CLASSES]].values)

## 10. Experimento principal (variante `completo`, semilla principal)

In [12]:
MAIN = train_run("completo", CFG["seed"])
MAIN_HIST = pd.read_json(Path(MAIN["run_dir"]) / "history.json")
y_test, P_plain, P_tta = load_test_predictions(MAIN["run_dir"])
print(f"Épocas: {MAIN['epochs_run']} (mejor {MAIN['best_epoch']}) | tiempo de entrenamiento: "
      f"{MAIN['train_time_s'] / 3600:.2f} h ({MAIN['sec_per_epoch']:.0f} s/época) | memoria GPU pico {MAIN['peak_gpu_mem_gb']:.1f} GB")
MAIN_HIST.drop(columns="lrs").tail()

  época 1: etapa ['cbam', 'head']
  [  1] loss 1.3337/1.1426 | acc mix 0.217 limpio 0.373 val 0.356 | F1 val 0.2727 * | 66s
  [  2] loss 1.2524/1.1310 | acc mix 0.231 limpio 0.356 val 0.358 | F1 val 0.2599 | 63s
  [  3] loss 1.2232/1.0895 | acc mix 0.249 limpio 0.397 val 0.374 | F1 val 0.2924 * | 62s
  [  4] loss 1.2146/1.0770 | acc mix 0.253 limpio 0.400 val 0.370 | F1 val 0.2727 | 63s
  [  5] loss 1.2030/1.0891 | acc mix 0.253 limpio 0.390 val 0.383 | F1 val 0.2978 * | 63s


KeyboardInterrupt: 

## 11. Métricas con intervalos y errores tipo I/II

* Intervalos de Wilson al 95 % para exactitud, y para precisión y sensibilidad (*recall*) de cada clase.
* F1 y **F2** (más peso a la sensibilidad) por clase y macro.
* Error **tipo I** por clase (uno contra el resto) = FP / (FP + TN) — tasa de falsos positivos.
  Error **tipo II** = FN / (FN + TP) = 1 − sensibilidad.

In [ ]:
def wilson(k, n, z=1.959964):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def global_metrics(y, pred):
    acc = accuracy_score(y, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    lo, hi = wilson(int((y == pred).sum()), len(y))
    return OrderedDict(accuracy=acc, acc_ci_low=lo, acc_ci_high=hi, precision_macro=pr, recall_macro=rc,
                       f1_macro=f1, f1_weighted=f1_score(y, pred, average="weighted", zero_division=0),
                       f2_macro=fbeta_score(y, pred, beta=2, average="macro", zero_division=0), n=len(y))

def per_class_table(y, pred):
    cm = confusion_matrix(y, pred, labels=range(NUM_CLASSES))
    f2 = fbeta_score(y, pred, beta=2, average=None, labels=range(NUM_CLASSES), zero_division=0)
    rows = []
    for k, c in enumerate(CLASSES):
        tp = cm[k, k]; fn = cm[k].sum() - tp; fp = cm[:, k].sum() - tp; tn = cm.sum() - tp - fn - fp
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        pl, ph = wilson(tp, tp + fp); rl, rh = wilson(tp, tp + fn)
        rows.append(OrderedDict(cls=c, support=int(tp + fn), TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn),
                                precision=prec, prec_ci_low=pl, prec_ci_high=ph,
                                recall=rec, rec_ci_low=rl, rec_ci_high=rh, f1=f1, f2=float(f2[k]),
                                type_I_FPR=fp / (fp + tn) if fp + tn else 0.0,
                                type_II_FNR=fn / (fn + tp) if fn + tp else 0.0))
    return pd.DataFrame(rows)

pred_plain, pred_tta = P_plain.argmax(1), P_tta.argmax(1)
G = pd.DataFrame({"sin TTA": global_metrics(y_test, pred_plain), "con TTA": global_metrics(y_test, pred_tta)})
PC_plain, PC_tta = per_class_table(y_test, pred_plain), per_class_table(y_test, pred_tta)

# test sin las imágenes duplicadas con train/val
keep = np.array([p not in test_dup_paths for p, _ in test_samples])
if (~keep).any():
    G["sin TTA (test sin dup.)"] = pd.Series(global_metrics(y_test[keep], pred_plain[keep]))
    G["con TTA (test sin dup.)"] = pd.Series(global_metrics(y_test[keep], pred_tta[keep]))
print(classification_report(y_test, pred_tta, target_names=CLASSES, digits=4, zero_division=0))
G.round(4)

In [ ]:
PC_tta.round(4)

## 12. Contrastes: McNemar exacto y bootstrap pareado (TTA frente a sin TTA)

Como se guardan las predicciones de cada imagen, ambas condiciones se comparan **sobre las mismas imágenes**:

* McNemar exacto (binomial sobre los pares discordantes *b* y *c*).
* Bootstrap pareado (remuestreo de imágenes) de la diferencia de exactitud y de F1 macro, con IC al 95 %.

In [ ]:
def mcnemar_exact(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b)); c = int(np.sum(~correct_a & correct_b))
    p = 1.0 if b + c == 0 else min(1.0, 2 * stats.binom.cdf(min(b, c), b + c, 0.5))
    return dict(b_only_A_correct=b, c_only_B_correct=c, p_value=p)

def fast_macro_f1(y, p):
    cm = np.bincount(y * NUM_CLASSES + p, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    tp = np.diag(cm).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.nan_to_num(tp / cm.sum(0)); rec = np.nan_to_num(tp / cm.sum(1))
        f1 = np.nan_to_num(2 * prec * rec / (prec + rec))
    return f1.mean()

def paired_bootstrap(y, pa, pb, B, seed=0):
    """Diferencia B - A en exactitud y F1 macro."""
    rng = np.random.RandomState(seed); n = len(y)
    d_acc, d_f1 = np.empty(B), np.empty(B)
    for i in range(B):
        idx = rng.randint(0, n, n)
        yy, a, b = y[idx], pa[idx], pb[idx]
        d_acc[i] = (b == yy).mean() - (a == yy).mean()
        d_f1[i] = fast_macro_f1(yy, b) - fast_macro_f1(yy, a)
    out = {}
    for name, d, obs in [("acc", d_acc, (pb == y).mean() - (pa == y).mean()),
                         ("f1_macro", d_f1, fast_macro_f1(y, pb) - fast_macro_f1(y, pa))]:
        out[name] = dict(diff=obs, ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                         p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    return out

MC = mcnemar_exact(pred_tta == y_test, pred_plain == y_test)    # A = TTA, B = sin TTA
BOOT = paired_bootstrap(y_test, pred_plain, pred_tta, CFG["n_bootstrap"], CFG["seed"])
print("McNemar exacto (b = sólo TTA acierta, c = sólo sin TTA acierta):", MC)
pd.DataFrame(BOOT).T.round(5)

## 12b. Mismas estadísticas sobre el checkpoint original del artículo

Evalúa `CFG["original_checkpoint"]` (el modelo cuyos números aparecen en el artículo) con este mismo test, TTA y estadísticas.
Así se pueden rellenar los IC, McNemar y bootstrap **de los números ya publicados** sin reentrenar.

In [ ]:
ORIG = None
ck = CFG.get("original_checkpoint")
if ck and Path(ck).exists():
    m = ResNet50CBAM(NUM_CLASSES, ("layer4",), pretrained=False).to(device)
    m.load_state_dict(torch.load(ck, map_location=device, weights_only=True))
    o0, ot, _ = predict(m, make_loader(test_samples, None, False, tta=True), tta=True)
    del m; torch.cuda.empty_cache()
    op, ott = o0.argmax(1), ot.argmax(1)
    ORIG = dict(G=pd.DataFrame({"sin TTA": global_metrics(y_test, op), "con TTA": global_metrics(y_test, ott)}),
                PC_tta=per_class_table(y_test, ott), MC=mcnemar_exact(ott == y_test, op == y_test),
                BOOT=paired_bootstrap(y_test, op, ott, CFG["n_bootstrap"], CFG["seed"]),
                vs_new=mcnemar_exact(pred_tta == y_test, ott == y_test))
    pd.DataFrame({"path": [p for p, _ in test_samples], "label": y_test, "pred": op, "pred_tta": ott}).to_csv(
        OUT / "preds_test_checkpoint_original.csv", index=False)
    print("McNemar TTA vs sin TTA (checkpoint original):", ORIG["MC"])
    print("McNemar modelo nuevo (TTA) vs original (TTA):", ORIG["vs_new"])
    display(ORIG["G"].round(4))
else:
    print("Sin checkpoint original (o no existe la ruta): se omite.")

## 13. Ablación

* **Por componentes:** `completo`, `base` (sin CBAM, entropía cruzada, sin MixUp), `sin_cbam`, `sin_focal`, `sin_mixup`.
* **Por posición de CBAM:** `cbam_layer2`, `cbam_layer3`, `cbam_layer3_4`, `cbam_layer2_3_4`.
* Cada variante con las semillas de `ablation_seeds`; se informa media ± DT de exactitud y F1 macro (con y sin TTA).
* McNemar exacto de cada variante frente a `completo`: por semilla emparejada (nº de semillas con p < 0,05) y sobre el
  conjunto promediado de semillas (media de probabilidades), con **corrección de Holm** entre variantes.

Viene desactivada (`RUN_ABLATION=False`): son 45 entrenamientos. Es reanudable: lo ya entrenado se reutiliza.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[i])); adj[i] = running
    return adj

ABL = None
if CFG["RUN_ABLATION"]:
    t_abl = time.time()
    for v in CFG["ablation_variants"]:
        for s in CFG["ablation_seeds"]:
            print(f"=== {v} | semilla {s} ===")
            train_run(v, s, verbose=False)
    print(f"Ablación: {(time.time() - t_abl) / 3600:.2f} h")

done = {}
for v in VARIANTS:
    for s in sorted(set(CFG["ablation_seeds"]) | {CFG["seed"]}):
        d = RUNS_DIR / f"{v}_s{s}"
        if (d / "meta.json").exists():
            done.setdefault(v, {})[s] = d

if "completo" in done and len(done) > 1:
    rows, per_seed = [], []
    ref = {s: load_test_predictions(d) for s, d in done["completo"].items()}
    for v, seeds in done.items():
        accs, f1s, accs_t, f1s_t, sig, ens_p = [], [], [], [], 0, []
        for s, d in seeds.items():
            y, p0, pt = load_test_predictions(d)
            a, at = p0.argmax(1), pt.argmax(1)
            meta = json.load(open(d / "meta.json"))
            accs.append((a == y).mean()); f1s.append(fast_macro_f1(y, a))
            accs_t.append((at == y).mean()); f1s_t.append(fast_macro_f1(y, at))
            r = dict(variant=v, seed=s, acc=accs[-1], f1_macro=f1s[-1], acc_tta=accs_t[-1], f1_macro_tta=f1s_t[-1],
                     epochs=meta["epochs_run"], train_h=meta["train_time_s"] / 3600)
            if v != "completo" and s in ref:
                r["mcnemar_p_vs_completo"] = mcnemar_exact(at == y, ref[s][2].argmax(1) == y)["p_value"]
                sig += r["mcnemar_p_vs_completo"] < 0.05
            per_seed.append(r); ens_p.append(pt)
        ens_pred = np.mean(ens_p, 0).argmax(1)
        rows.append(dict(variant=v, n_seeds=len(seeds), acc_mean=np.mean(accs), acc_sd=np.std(accs, ddof=1) if len(accs) > 1 else 0,
                         f1_mean=np.mean(f1s), f1_sd=np.std(f1s, ddof=1) if len(f1s) > 1 else 0,
                         acc_tta_mean=np.mean(accs_t), acc_tta_sd=np.std(accs_t, ddof=1) if len(accs_t) > 1 else 0,
                         f1_tta_mean=np.mean(f1s_t), f1_tta_sd=np.std(f1s_t, ddof=1) if len(f1s_t) > 1 else 0,
                         seeds_p_lt_005=sig if v != "completo" else None, _ens_pred=ens_pred))
    ref_ens = [r for r in rows if r["variant"] == "completo"][0]["_ens_pred"]
    others = [r for r in rows if r["variant"] != "completo"]
    pv = [mcnemar_exact(r["_ens_pred"] == y_test, ref_ens == y_test)["p_value"] for r in others]
    for r, p, ph in zip(others, pv, holm(pv)):
        r["mcnemar_p_ens"], r["mcnemar_p_holm"] = p, ph
    ABL = pd.DataFrame(rows).drop(columns="_ens_pred")
    ABL["variant"] = pd.Categorical(ABL.variant, [v for v in VARIANTS if v in done]); ABL = ABL.sort_values("variant")
    ABL_SEEDS = pd.DataFrame(per_seed)
    ABL.to_csv(OUT / "ablacion_resumen.csv", index=False); ABL_SEEDS.to_csv(OUT / "ablacion_por_semilla.csv", index=False)
    display(ABL.round(4))
else:
    print("Ablación no ejecutada (o sólo hay 'completo').")

## 14. Grad-CAM: aciertos y errores por clase

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = layer.register_forward_hook(self._fwd)
    def _fwd(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))
    def __call__(self, x, cls):
        self.model.zero_grad()
        x = x.clone().requires_grad_(True)       # garantiza gradiente aunque haya capas congeladas
        out = self.model(x)
        out[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min()
        return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), out.softmax(1)[0].detach().cpu().numpy()
    def remove(self):
        self.h.remove()

cam_model = ResNet50CBAM(NUM_CLASSES, ("layer4",), pretrained=False).to(device)
cam_model.load_state_dict(torch.load(Path(MAIN["run_dir"]) / "best.pth", map_location=device, weights_only=True))
cam_model.eval()
gc = GradCAM(cam_model, cam_model.gradcam_layer())

conf = P_plain.max(1)
cam_examples = []   # (fila de clase, tipo, índice de test)
for k in range(NUM_CLASSES):
    ok = np.where((y_test == k) & (pred_plain == k))[0]
    bad = np.where((y_test == k) & (pred_plain != k))[0]
    ok = ok[np.argsort(-conf[ok])][:2]; bad = bad[np.argsort(-conf[bad])][:2]
    cam_examples += [(k, "correct", i) for i in ok] + [(k, "error", i) for i in bad]

CAMS = {}
for k, kind, i in cam_examples:
    im = Image.open(test_samples[i][0]).convert("RGB")
    x = eval_transform(im).unsqueeze(0).to(device)
    cam, prob = gc(x, int(pred_plain[i]))
    CAMS[(k, kind, i)] = (np.asarray(im.convert("L").resize((S, S))), cam, prob)
gc.remove(); del cam_model; torch.cuda.empty_cache()
print("Mapas Grad-CAM calculados:", len(CAMS))

## 15. Salidas para Overleaf: figuras (PNG + PDF, 300 ppp)

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "figure.dpi": 110,
                     "savefig.bbox": "tight", "pdf.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False})
C_BLUE, C_ORANGE, C_GREEN, C_GREY = "#2b6cb0", "#dd6b20", "#2f855a", "#718096"

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300)
    plt.show(); plt.close(fig)

def stage_lines(ax):
    for s in CFG["stages"][1:]:
        if s["start_epoch"] < len(MAIN_HIST):
            ax.axvline(s["start_epoch"] + 0.5, color=C_GREY, ls=":", lw=1)

# Fig. 1 — distribución de clases
fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(NUM_CLASSES)
ax.bar(x, counts(train_orig), color=C_BLUE, label="Train (original)")
ax.bar(x, counts(train_gen), bottom=counts(train_orig), color=C_BLUE, alpha=0.35, label="Train (generated)")
ax.set_xticks(x); ax.set_xticklabels(CLASSES); ax.set_ylabel("Images")
ax2 = ax.twinx(); ax2.spines["top"].set_visible(False)
ax2.plot(x, counts(test_samples), "o-", color=C_ORANGE, label="Test (natural)"); ax2.set_ylabel("Test images")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="upper left")
ax.set_title("Class distribution: balanced training set vs. natural test set")
savefig(fig, "fig1_distribucion_clases")

# Fig. 2 — pérdidas
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(MAIN_HIST.epoch, MAIN_HIST.train_loss, color=C_BLUE, label="Train (MixUp)")
ax.plot(MAIN_HIST.epoch, MAIN_HIST.val_loss, color=C_ORANGE, label="Validation")
ax.axvline(MAIN["best_epoch"], color="k", lw=0.8, ls="--", label=f"Best epoch ({MAIN['best_epoch']})")
stage_lines(ax); ax.set_xlabel("Epoch"); ax.set_ylabel("Focal loss"); ax.legend(frameon=False)
ax.set_title("Training and validation loss (dotted: unfreezing stages)")
savefig(fig, "fig2_curvas_perdida")

# Fig. 3 — exactitud con y sin mezcla
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(MAIN_HIST.epoch, MAIN_HIST.train_acc_mixup, color=C_BLUE, alpha=0.5, label="Train, MixUp batches")
ax.plot(MAIN_HIST.epoch, MAIN_HIST.train_acc_clean, color=C_BLUE, label="Train, clean (eval mode)")
ax.plot(MAIN_HIST.epoch, MAIN_HIST.val_acc, color=C_ORANGE, label="Validation")
ax.plot(MAIN_HIST.epoch, MAIN_HIST.val_f1_macro, color=C_GREEN, ls="--", label="Validation macro-F1")
ax.axvline(MAIN["best_epoch"], color="k", lw=0.8, ls="--"); stage_lines(ax)
ax.set_xlabel("Epoch"); ax.set_ylabel("Score"); ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=8)
ax.set_title("Accuracy: MixUp accuracy is not comparable with validation; clean accuracy is")
savefig(fig, "fig3_curvas_exactitud")

# Fig. 4 y 5 — matrices de confusión normalizadas
def plot_cm(pred, title, name):
    cm = confusion_matrix(y_test, pred, labels=range(NUM_CLASSES))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                    color="white" if cmn[i, j] > 0.55 else "black")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    savefig(fig, name)

plot_cm(pred_plain, f"Confusion matrix, no TTA (acc {G.loc['accuracy', 'sin TTA']:.3f})", "fig4_matriz_confusion_sin_tta")
plot_cm(pred_tta, f"Confusion matrix, 3-view TTA (acc {G.loc['accuracy', 'con TTA']:.3f})", "fig5_matriz_confusion_tta")

# Fig. 6 — precisión y sensibilidad por clase con IC de Wilson
fig, ax = plt.subplots(figsize=(6.5, 3.2))
w = 0.38
for off, col, lab, c in [(-w / 2, "precision", "Precision", C_BLUE), (w / 2, "recall", "Recall", C_ORANGE)]:
    lo, hi = ("prec_ci_low", "prec_ci_high") if col == "precision" else ("rec_ci_low", "rec_ci_high")
    v = PC_tta[col].values
    ax.bar(x + off, v, w, color=c, label=lab)
    ax.errorbar(x + off, v, yerr=[v - PC_tta[lo].values, PC_tta[hi].values - v], fmt="none", ecolor="k", lw=0.8, capsize=2)
ax.plot(x, PC_tta.f1, "D", color="k", ms=4, label="F1")
ax.set_xticks(x); ax.set_xticklabels([f"{c}\n(n={n})" for c, n in zip(CLASSES, PC_tta.support)], fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.legend(frameon=False, ncol=3, fontsize=8)
ax.set_title("Per-class precision and recall with 95% Wilson intervals (TTA)")
savefig(fig, "fig6_metricas_por_clase_wilson")

# Fig. 7 — ablación
if ABL is not None:
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    xa = np.arange(len(ABL))
    ax.bar(xa, ABL.f1_tta_mean, yerr=ABL.f1_tta_sd, color=[C_ORANGE if v == "completo" else C_BLUE for v in ABL.variant],
           capsize=3, alpha=0.85)
    for i, v in enumerate(ABL.variant):
        pts = ABL_SEEDS[ABL_SEEDS.variant == v].f1_macro_tta
        ax.plot(np.full(len(pts), i) + np.linspace(-0.15, 0.15, len(pts)), pts, ".", color="k", ms=4)
        r = ABL.iloc[i]
        if v != "completo" and not pd.isna(r.get("mcnemar_p_holm", np.nan)) and r.mcnemar_p_holm < 0.05:
            ax.text(i, r.f1_tta_mean + r.f1_tta_sd + 0.005, "*", ha="center", fontsize=12)
    ax.set_xticks(xa); ax.set_xticklabels(ABL.variant, rotation=30, ha="right")
    lo = max(0, ABL_SEEDS.f1_macro_tta.min() - 0.05)
    ax.set_ylim(lo, min(1, ABL_SEEDS.f1_macro_tta.max() + 0.04)); ax.set_ylabel("Test macro-F1 (TTA)")
    ax.set_title(f"Ablation: mean ± SD over seeds (dots); * Holm-adjusted McNemar p < 0.05 vs. full")
    savefig(fig, "fig7_ablacion")
else:
    print("Fig. 7 omitida: la ablación no se ha ejecutado.")

# Fig. 8 — Grad-CAM
cols = 4
fig, axes = plt.subplots(NUM_CLASSES, cols, figsize=(cols * 1.55, NUM_CLASSES * 1.65))
for k in range(NUM_CLASSES):
    items = [e for e in cam_examples if e[0] == k]
    items = [e for e in items if e[1] == "correct"][:2] + [e for e in items if e[1] == "error"][:2]
    for j in range(cols):
        ax = axes[k, j]; ax.axis("off")
        if j < len(items):
            _, kind, i = items[j]
            g, cam, prob = CAMS[items[j]]
            ax.imshow(g, cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.45)
            p = int(pred_plain[i])
            ax.set_title(f"{'✓' if kind == 'correct' else '✗'} pred {CLASSES[p]} ({prob[p]:.2f})", fontsize=6.5,
                         color=C_GREEN if kind == "correct" else "#c53030")
    axes[k, 0].text(-0.15, 0.5, CLASSES[k], transform=axes[k, 0].transAxes, rotation=90, va="center", ha="right", fontsize=8)
fig.suptitle("Grad-CAM on the CBAM output: two correct and two misclassified test images per true class", fontsize=8)
fig.tight_layout()
savefig(fig, "fig8_gradcam")
print("Figuras en", FIG_DIR)

### 15b. Tablas LaTeX (booktabs) y resumen para los [COMPLETAR]

In [ ]:
def esc(s):
    return str(s).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&").replace("#", r"\#")

def latex_table(df, name, caption, label, col_fmt=None):
    cols = list(df.columns); col_fmt = col_fmt or "l" + "r" * (len(cols) - 1)
    L = [r"\begin{table}[ht]", r"\centering", r"\small", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
         rf"\begin{{tabular}}{{{col_fmt}}}", r"\toprule", " & ".join(esc(c) for c in cols) + r" \\", r"\midrule"]
    for _, row in df.iterrows():
        L.append(" & ".join(v if isinstance(v, str) and "$" in v else esc(v) for v in row.values) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(L), encoding="utf-8")

f3 = lambda v: f"{v:.3f}"
ci = lambda v, lo, hi: f"{v:.3f} [{lo:.3f}, {hi:.3f}]"
pm = lambda m, s: f"{m:.3f} $\\pm$ {s:.3f}"

# Tabla 3 — entorno y coste
t_env = pd.DataFrame([
    ("GPU", f"{ENV['gpu']} ({ENV['gpu_mem_gb']} GB), driver {ENV['driver']}"), ("CUDA / cuDNN", f"{ENV['cuda']} / {ENV['cudnn']}"),
    ("Python / PyTorch / torchvision", f"{ENV['python']} / {ENV['torch']} / {ENV['torchvision']}"),
    ("scikit-learn / SciPy / NumPy", f"{ENV['sklearn']} / {ENV['scipy']} / {ENV['numpy']}"),
    ("Random seed", str(CFG["seed"])), ("Mixed precision (AMP)", "yes" if AMP else "no"),
    ("Epochs run (best epoch)", f"{MAIN['epochs_run']} ({MAIN['best_epoch']})"),
    ("Training time", f"{MAIN['train_time_s'] / 3600:.2f} h ({MAIN['sec_per_epoch']:.0f} s/epoch)"),
    ("Test inference, 3-view TTA", f"{MAIN['test_infer_time_tta_s']:.1f} s for {len(test_samples)} images"),
    ("Peak GPU memory", f"{MAIN['peak_gpu_mem_gb']:.1f} GB"),
    ("Parameters (backbone / CBAM / head)", f"{PARAMS['backbone']:,} / {PARAMS['cbam']:,} / {PARAMS['head']:,}"),
], columns=["Item", "Value"])
latex_table(t_env, "tabla3_entorno", "Computational environment and training cost.", "tab:environment", "ll")

# Tabla 4 — hiperparámetros
st = CFG["stages"]
t_hp = pd.DataFrame([
    ("Input size", f"{S}x{S}, grayscale replicated to 3 channels"), ("Batch size", CFG["batch_size"]),
    ("Max epochs / early-stopping patience", f"{CFG['max_epochs']} / {CFG['patience']} (val macro-F1)"),
    ("Optimizer", f"AdamW, weight decay {CFG['weight_decay']}"),
    ("LR scheduler", f"ReduceLROnPlateau (factor {CFG['plateau_factor']}, patience {CFG['plateau_patience']})"),
    ("Focal loss gamma", CFG["focal_gamma"]), ("MixUp alpha / probability", f"{CFG['mixup_alpha']} / {CFG['mixup_prob']}"),
    ("Dropout (head)", f"{CFG['dropout1']} / {CFG['dropout2']}"), ("CBAM reduction / kernel", f"{CFG['cbam_reduction']} / {CFG['cbam_kernel']}"),
] + [(f"Stage {i + 1} (from epoch {s['start_epoch'] + 1}): LR", ", ".join(f"{k} {v:g}" for k, v in s["lr"].items()))
     for i, s in enumerate(st)], columns=["Hyperparameter", "Value"])
latex_table(t_hp, "tabla4_hiperparametros", "Training hyperparameters.", "tab:hyperparameters", "ll")

# Datos y duplicados
latex_table(data_table.astype(str), "tabla_datos", "Images per class: original and generated training images, validation and natural test distribution.", "tab:data")
latex_table(dup_summary.astype(str) if len(dup_summary) else pd.DataFrame([["--"] * 4], columns=["split_a", "split_b", "pairs", "exact_md5"]),
            "tabla_duplicados", f"Near-duplicate pairs across partitions (perceptual hash, Hamming distance $\\leq$ {CFG['dup_hamming']}).", "tab:duplicates")

# Métricas globales
def gl_rows(Gd):
    out = []
    for col in Gd.columns:
        g = Gd[col]
        out.append((col.replace("sin TTA", "No TTA").replace("con TTA", "3-view TTA").replace("test sin dup.", "deduplicated test"),
                    ci(g.accuracy, g.acc_ci_low, g.acc_ci_high), f3(g.precision_macro), f3(g.recall_macro),
                    f3(g.f1_macro), f3(g.f2_macro), f3(g.f1_weighted), str(int(g.n))))
    return pd.DataFrame(out, columns=["Setting", "Accuracy [95% CI]", "Macro P", "Macro R", "Macro F1", "Macro F2", "Weighted F1", "n"])
latex_table(gl_rows(G), "tabla_metricas_globales",
            f"Test performance. Exact McNemar TTA vs. no TTA: b={MC['b_only_A_correct']}, c={MC['c_only_B_correct']}, p={MC['p_value']:.4f}. "
            f"Paired bootstrap ($B$={CFG['n_bootstrap']}) $\\Delta$acc={BOOT['acc']['diff']:.4f} [{BOOT['acc']['ci_low']:.4f}, {BOOT['acc']['ci_high']:.4f}], "
            f"$\\Delta$macro-F1={BOOT['f1_macro']['diff']:.4f} [{BOOT['f1_macro']['ci_low']:.4f}, {BOOT['f1_macro']['ci_high']:.4f}].",
            "tab:global")

# Por clase
def pc_rows(P):
    return pd.DataFrame([(r.cls, str(r.support), ci(r.precision, r.prec_ci_low, r.prec_ci_high), ci(r.recall, r.rec_ci_low, r.rec_ci_high),
                          f3(r.f1), f3(r.f2), f3(r.type_I_FPR), f3(r.type_II_FNR)) for r in P.itertuples()],
                        columns=["Class", "n", "Precision [95% CI]", "Recall [95% CI]", "F1", "F2", "Type I (FPR)", "Type II (FNR)"])
latex_table(pc_rows(PC_tta), "tabla_por_clase_tta", "Per-class test metrics with 3-view TTA (one-vs-rest; Wilson 95\\% intervals).", "tab:perclass")
latex_table(pc_rows(PC_plain), "tabla_por_clase_sin_tta", "Per-class test metrics without TTA (one-vs-rest; Wilson 95\\% intervals).", "tab:perclass-notta")

if ORIG is not None:
    latex_table(gl_rows(ORIG["G"]), "tabla_metricas_checkpoint_original",
                f"Test performance of the published checkpoint. Exact McNemar TTA vs. no TTA: p={ORIG['MC']['p_value']:.4f}.", "tab:global-original")
    latex_table(pc_rows(ORIG["PC_tta"]), "tabla_por_clase_checkpoint_original", "Per-class metrics of the published checkpoint (TTA).", "tab:perclass-original")

if ABL is not None:
    t_abl = pd.DataFrame([(r.variant, str(r.n_seeds), pm(r.acc_mean, r.acc_sd), pm(r.f1_mean, r.f1_sd), pm(r.acc_tta_mean, r.acc_tta_sd),
                           pm(r.f1_tta_mean, r.f1_tta_sd),
                           "--" if r.variant == "completo" else f"{r.mcnemar_p_holm:.4f}",
                           "--" if r.variant == "completo" else f"{int(r.seeds_p_lt_005)}/{r.n_seeds}")
                          for r in ABL.itertuples()],
                         columns=["Variant", "Seeds", "Acc", "Macro F1", "Acc (TTA)", "Macro F1 (TTA)", "McNemar p (Holm)", "Seeds p<0.05"])
    latex_table(t_abl, "tabla_ablacion", "Ablation study (mean $\\pm$ SD over seeds). McNemar on seed-averaged TTA predictions vs. the full model, Holm-corrected.", "tab:ablation")

G.to_csv(OUT / "metricas_globales.csv"); PC_tta.to_csv(OUT / "metricas_por_clase_tta.csv", index=False)
PC_plain.to_csv(OUT / "metricas_por_clase_sin_tta.csv", index=False)
MAIN_HIST.to_csv(OUT / "historial_entrenamiento.csv", index=False)
shutil.copy2(Path(MAIN["run_dir"]) / "preds_test.csv", OUT / "preds_test_modelo_principal.csv")
print("Tablas:", sorted(p.name for p in TAB_DIR.glob("*.tex")))

In [ ]:
def r4(v):
    return None if v is None or (isinstance(v, float) and math.isnan(v)) else round(float(v), 4)

g0, gt = G["sin TTA"], G["con TTA"]
SUMMARY = OrderedDict(
    smoke=SMOKE,
    environment=ENV,
    params=PARAMS,
    data=dict(validation=VAL_SOURCE, train_original=int(len(train_orig)), train_generated=int(len(train_gen)),
              val=int(len(val_samples)), test=int(len(test_samples)), test_with_duplicate_in_train_or_val=len(test_dup_paths),
              duplicate_pairs={f"{r.split_a}-{r.split_b}": int(r.pairs) for r in dup_summary.itertuples()}),
    training=dict(epochs_run=MAIN["epochs_run"], best_epoch=MAIN["best_epoch"], best_val_f1=r4(MAIN["best_val_f1"]),
                  train_time_h=r4(MAIN["train_time_s"] / 3600), sec_per_epoch=r4(MAIN["sec_per_epoch"]),
                  peak_gpu_mem_gb=r4(MAIN["peak_gpu_mem_gb"]),
                  final_train_acc_clean=r4(MAIN_HIST.train_acc_clean.iloc[MAIN["best_epoch"] - 1]),
                  final_train_acc_mixup=r4(MAIN_HIST.train_acc_mixup.iloc[MAIN["best_epoch"] - 1]),
                  val_acc_at_best=r4(MAIN_HIST.val_acc.iloc[MAIN["best_epoch"] - 1])),
    test_no_tta={k: r4(v) for k, v in g0.items()}, test_tta={k: r4(v) for k, v in gt.items()},
    mcnemar_tta_vs_no_tta=MC, bootstrap_tta_minus_no_tta={k: {kk: r4(vv) for kk, vv in v.items()} for k, v in BOOT.items()},
    per_class_tta=PC_tta.set_index("cls")[["precision", "recall", "f1", "f2", "type_I_FPR", "type_II_FNR"]].round(4).to_dict("index"),
)
if ORIG is not None:
    SUMMARY["published_checkpoint"] = dict(test_no_tta={k: r4(v) for k, v in ORIG["G"]["sin TTA"].items()},
                                           test_tta={k: r4(v) for k, v in ORIG["G"]["con TTA"].items()},
                                           mcnemar_tta_vs_no_tta=ORIG["MC"], mcnemar_new_vs_published=ORIG["vs_new"],
                                           bootstrap={k: {kk: r4(vv) for kk, vv in v.items()} for k, v in ORIG["BOOT"].items()})
if ABL is not None:
    SUMMARY["ablation"] = ABL.round(4).to_dict("records")
json.dump(SUMMARY, open(OUT / "resumen_COMPLETAR.json", "w"), indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

L = ["# Valores para los [COMPLETAR] del artículo", "",
     "> " + ("**SMOKE: datos sintéticos, NO usar estos números.**" if SMOKE else f"Generado el {time.strftime('%Y-%m-%d %H:%M')}"), "",
     "## Entorno y coste (Tabla 3)",
     f"- GPU: {ENV['gpu']} ({ENV['gpu_mem_gb']} GB), driver {ENV['driver']}, CUDA {ENV['cuda']}, cuDNN {ENV['cudnn']}",
     f"- Python {ENV['python']}, PyTorch {ENV['torch']}, torchvision {ENV['torchvision']}; semilla {CFG['seed']}; AMP {AMP}",
     f"- Épocas: {MAIN['epochs_run']} (mejor: {MAIN['best_epoch']}); tiempo de entrenamiento: {MAIN['train_time_s'] / 3600:.2f} h ({MAIN['sec_per_epoch']:.0f} s/época)",
     f"- Parámetros: backbone {PARAMS['backbone']:,}, CBAM {PARAMS['cbam']:,}, cabeza {PARAMS['head']:,}, total {PARAMS['total']:,}",
     "", "## Datos",
     f"- Validación: {VAL_SOURCE}; train originales {len(train_orig)}, generadas {len(train_gen)} "
     f"({100 * len(train_gen) / max(1, len(train_samples)):.1f} % del train balanceado); val {len(val_samples)}; test {len(test_samples)}",
     f"- Imágenes de test con duplicado en train/val: {len(test_dup_paths)} " + str(SUMMARY["data"]["duplicate_pairs"]),
     "", "## Entrenamiento",
     f"- Exactitud de entrenamiento sin mezcla (mejor época): {SUMMARY['training']['final_train_acc_clean']}; "
     f"con MixUp: {SUMMARY['training']['final_train_acc_mixup']}; validación: {SUMMARY['training']['val_acc_at_best']}",
     "", "## Test (modelo reentrenado)",
     f"- Sin TTA: acc {g0.accuracy:.4f} [IC95 {g0.acc_ci_low:.4f}–{g0.acc_ci_high:.4f}], F1 macro {g0.f1_macro:.4f}, F2 macro {g0.f2_macro:.4f}",
     f"- Con TTA: acc {gt.accuracy:.4f} [IC95 {gt.acc_ci_low:.4f}–{gt.acc_ci_high:.4f}], F1 macro {gt.f1_macro:.4f}, F2 macro {gt.f2_macro:.4f}",
     f"- McNemar exacto TTA vs sin TTA: b={MC['b_only_A_correct']}, c={MC['c_only_B_correct']}, p={MC['p_value']:.4g}",
     f"- Bootstrap pareado (B={CFG['n_bootstrap']}): Δacc={BOOT['acc']['diff']:.4f} [{BOOT['acc']['ci_low']:.4f}, {BOOT['acc']['ci_high']:.4f}]; "
     f"ΔF1={BOOT['f1_macro']['diff']:.4f} [{BOOT['f1_macro']['ci_low']:.4f}, {BOOT['f1_macro']['ci_high']:.4f}]"]
if ORIG is not None:
    o0, ot = ORIG["G"]["sin TTA"], ORIG["G"]["con TTA"]
    L += ["", "## Checkpoint publicado (números del artículo)",
          f"- Sin TTA: acc {o0.accuracy:.4f} [{o0.acc_ci_low:.4f}–{o0.acc_ci_high:.4f}], F1 macro {o0.f1_macro:.4f}",
          f"- Con TTA: acc {ot.accuracy:.4f} [{ot.acc_ci_low:.4f}–{ot.acc_ci_high:.4f}], F1 macro {ot.f1_macro:.4f}",
          f"- McNemar TTA vs sin TTA: p={ORIG['MC']['p_value']:.4g}; Δacc bootstrap {ORIG['BOOT']['acc']['diff']:.4f} "
          f"[{ORIG['BOOT']['acc']['ci_low']:.4f}, {ORIG['BOOT']['acc']['ci_high']:.4f}]"]
L += ["", "## Por clase (TTA)", "", "| Clase | n | Precisión [IC95] | Sensibilidad [IC95] | F1 | F2 | Tipo I | Tipo II |", "|---|---|---|---|---|---|---|---|"]
L += [f"| {r.cls} | {r.support} | {r.precision:.3f} [{r.prec_ci_low:.3f}, {r.prec_ci_high:.3f}] | {r.recall:.3f} [{r.rec_ci_low:.3f}, {r.rec_ci_high:.3f}] "
      f"| {r.f1:.3f} | {r.f2:.3f} | {r.type_I_FPR:.3f} | {r.type_II_FNR:.3f} |" for r in PC_tta.itertuples()]
if ABL is not None:
    L += ["", "## Ablación (media ± DT)", "", "| Variante | Semillas | Acc TTA | F1 TTA | p Holm |", "|---|---|---|---|---|"]
    L += [f"| {r.variant} | {r.n_seeds} | {r.acc_tta_mean:.4f} ± {r.acc_tta_sd:.4f} | {r.f1_tta_mean:.4f} ± {r.f1_tta_sd:.4f} | "
          f"{'—' if r.variant == 'completo' else f'{r.mcnemar_p_holm:.4g}'} |" for r in ABL.itertuples()]
L += ["", "## Archivos", f"- Figuras: `{FIG_DIR}`", f"- Tablas LaTeX: `{TAB_DIR}`", f"- Checkpoints y datos generados: `{WORK}`"]
(OUT / "resumen_COMPLETAR.md").write_text("\n".join(L), encoding="utf-8")
print("\n".join(L))

## 16. Detector de duplicados: sensibilidad al umbral y revisión manual

La sección 5 sólo marca duplicados con dHash de 64 bits y distancia de Hamming 0. Aquí se comprueba cuánto depende
el resultado de ese umbral:

* Dos hashes independientes: el **dHash** de la sección 5 y un **pHash-DCT** (DCT 2D de 32x32, 8x8 coeficientes de
  baja frecuencia). Ambos se calculan también sobre la imagen **volteada**, para detectar copias especulares.
* Para cada imagen de test (y de validación) se busca su vecino más cercano en train+val. Se cuentan las imágenes con
  un vecino dentro de cada umbral (0-12 bits).
* **Revisión manual:** muestra estratificada por banda de distancia, 20 pares por banda y por hash (360 pares). Las
  láminas están en `figuras/revision_duplicados/` y el veredicto en la columna `verdict` de
  `revision_duplicados_muestra.csv`: `same` = misma fotografía de origen (admite recorte, escala, brillo, marca de agua
  o espejo), `different` u `uncertain`.

In [ ]:
#@setup:dup
from scipy.fftpack import dct as _dct

DUP_THRESHOLDS = [0, 1, 2, 3, 4, 6, 8, 10, 12]
REVIEW_PER_BAND = 20

def _bits(b):
    return int("".join("1" if v else "0" for v in b.flatten()), 2)

def dhash_img(im):
    a = np.asarray(im.resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return _bits(a[:, 1:] > a[:, :-1])

def dct_phash_img(im):
    """pHash clásico: DCT 2D de la imagen 32x32; bits de los 8x8 coeficientes de baja frecuencia frente a su mediana."""
    a = np.asarray(im.resize((32, 32), Image.LANCZOS), dtype=np.float64)
    d = _dct(_dct(a, axis=0, norm="ortho"), axis=1, norm="ortho")[:8, :8]
    return _bits(d > np.median(d))

def extra_hashes(h, name):
    """Añade dHash de la imagen volteada y pHash-DCT (normal y volteada). Caché en hex en WORK_DIR."""
    cols = ["dhash_flip", "phash_dct", "phash_dct_flip"]
    cache = WORK / f"hashes_extra_{name}.csv"
    if cache.exists() and not SMOKE:
        e = pd.read_csv(cache, dtype=str)
        if list(e.path) == list(h.path):
            for c in cols:
                h[c] = np.array([int(v, 16) for v in e[c]], dtype=np.uint64)
            return h
    vals = {c: [] for c in cols}
    for p in h.path:
        im = Image.open(p).convert("L"); fl = im.transpose(Image.FLIP_LEFT_RIGHT)
        vals["dhash_flip"].append(dhash_img(fl)); vals["phash_dct"].append(dct_phash_img(im))
        vals["phash_dct_flip"].append(dct_phash_img(fl))
    for c in cols:
        h[c] = np.array(vals[c], dtype=np.uint64)
    pd.DataFrame({"path": h.path, **{c: [f"{int(v):016x}" for v in h[c]] for c in cols}}).to_csv(cache, index=False)
    return h

def hamming_matrix(a, b):
    x = np.bitwise_xor(np.asarray(a, dtype=np.uint64)[:, None], np.asarray(b, dtype=np.uint64)[None, :])
    return POPCOUNT[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1)

def hash_cols(kind):
    return ("phash", "dhash_flip") if kind == "d" else ("phash_dct", "phash_dct_flip")

def nearest(hq, hr, kind, same=False, chunk=256):
    """Distancia mínima (y vecino) de cada imagen de hq a hr, considerando también la versión especular.
    kind='d' (dHash, columna 'phash' de la sección 5) o 'p' (pHash-DCT). same=True excluye la propia imagen."""
    c, cf = hash_cols(kind)
    q, qf, r = hq[c].values, hq[cf].values, hr[c].values
    dmin, arg, mirr = np.empty(len(q), int), np.empty(len(q), int), np.empty(len(q), bool)
    for s in range(0, len(q), chunk):
        d0, d1 = hamming_matrix(q[s:s + chunk], r), hamming_matrix(qf[s:s + chunk], r)
        if same:
            ii = np.arange(d0.shape[0]); d0[ii, s + ii] = 99; d1[ii, s + ii] = 99
        d = np.minimum(d0, d1)
        a = d.argmin(1); ii = np.arange(len(a))
        dmin[s:s + chunk], arg[s:s + chunk], mirr[s:s + chunk] = d[ii, a], a, d1[ii, a] < d0[ii, a]
    return dmin, arg, mirr

def pair_dist(ra, rb, kind):
    c, cf = hash_cols(kind)
    return int(min(hamming_matrix([ra[c]], [rb[c]])[0, 0], hamming_matrix([ra[cf]], [rb[c]])[0, 0]))

t0 = time.time()
for k in H:
    H[k] = extra_hashes(H[k], k)
H_trval = pd.concat([H["train"].assign(split="train"), H["val"].assign(split="val")], ignore_index=True)

NN = {}   # (consulta, referencia, hash) -> (distancia, índice del vecino, especular)
for kind in ("d", "p"):
    NN[("test", "train+val", kind)] = nearest(H["test"], H_trval, kind)
    NN[("val", "train", kind)] = nearest(H["val"], H["train"], kind)
    NN[("val", "test", kind)] = nearest(H["val"], H["test"], kind)
    NN[("train", "train", kind)] = nearest(H["train"], H["train"], kind, same=True)
print(f"Hashes extra y vecinos más cercanos en {time.time() - t0:.0f} s")

sens_rows = []
for (qn, rn, kind), (d, _, m) in NN.items():
    for t in DUP_THRESHOLDS:
        hit = d <= t
        sens_rows.append(dict(query=qn, reference=rn, hash="dHash" if kind == "d" else "pHash-DCT", threshold=t,
                              images_with_match=int(hit.sum()), pct=100 * hit.mean(), mirrored=int((hit & m).sum())))
DUP_SENS = pd.DataFrame(sens_rows)
DUP_SENS.to_csv(OUT / "duplicados_sensibilidad_umbral.csv", index=False)
display(DUP_SENS.pivot_table(index=["hash", "threshold"], columns=["query", "reference"], values="images_with_match"))

In [ ]:
# Muestras estratificadas por banda de distancia (una por hash): cada imagen de test junto a su vecino más cercano
# en train+val según ese hash. Se guardan láminas para la revisión visual y un CSV con la columna 'verdict' vacía.
REVIEW_BANDS = {"d": [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 6), (7, 8), (9, 10), (11, 12)],
                "p": [(2, 2), (4, 4), (6, 6), (8, 8), (10, 10), (12, 12), (14, 14), (16, 18), (0, 0)]}
REV_DIR = FIG_DIR / "revision_duplicados"; REV_DIR.mkdir(exist_ok=True)

def review_sample(kind, seed):
    dist, arg, mirr = NN[("test", "train+val", kind)]
    rng = np.random.RandomState(seed); rows = []
    for lo, hi in REVIEW_BANDS[kind]:
        idx = np.where((dist >= lo) & (dist <= hi))[0]
        for i in sorted(rng.choice(idx, min(REVIEW_PER_BAND, len(idx)), replace=False)):
            q, r = H["test"].iloc[i], H_trval.iloc[arg[i]]
            rows.append(dict(sample=f"{kind}Hash", band=f"{lo}" if lo == hi else f"{lo}-{hi}", test_path=q.path,
                             ref_path=r.path, ref_split=r.split, label_test=CLASSES[q.label], label_ref=CLASSES[r.label],
                             dhash_d=pair_dist(q, r, "d"), phash_d=pair_dist(q, r, "p"), mirrored=bool(mirr[i]),
                             md5_equal=q.md5 == r.md5, verdict=""))
    return rows

def review_sheet(grp, title, path, cols=5):
    rows = math.ceil(len(grp) / cols)
    fig, axes = plt.subplots(rows, 2 * cols, figsize=(2 * cols * 1.5, rows * 1.85), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for j, r in enumerate(grp.itertuples()):
        ax_a, ax_b = axes[j // cols, 2 * (j % cols)], axes[j // cols, 2 * (j % cols) + 1]
        ax_a.imshow(Image.open(r.test_path).convert("L"), cmap="gray", vmin=0, vmax=255)
        ax_b.imshow(Image.open(r.ref_path).convert("L"), cmap="gray", vmin=0, vmax=255)
        ax_a.set_title(f"#{r.id} {r.label_test[:3]}", fontsize=7)
        ax_b.set_title(f"{r.label_ref[:3]} d{r.dhash_d}/p{r.phash_d}{' M' if r.mirrored else ''}", fontsize=7)
    fig.suptitle(title, fontsize=9)
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

REVIEW = pd.DataFrame(review_sample("d", CFG["seed"]) + review_sample("p", CFG["seed"] + 1))
REVIEW.insert(0, "id", range(len(REVIEW)))
rev_csv = OUT / "revision_duplicados_muestra.csv"
if rev_csv.exists():   # conserva los veredictos ya anotados a mano (por par de imágenes)
    old = pd.read_csv(rev_csv, keep_default_na=False).set_index(["sample", "test_path", "ref_path"])
    for c in ("verdict", "reviewer"):
        if c in old:
            REVIEW[c] = [old[c].get((r["sample"], r.test_path, r.ref_path), "") for _, r in REVIEW.iterrows()]
REVIEW.to_csv(rev_csv, index=False)
for (smp, band), grp in REVIEW.groupby(["sample", "band"], sort=False):
    review_sheet(grp, f"{smp} band {band}: test (left) vs nearest train/val image (right); d/p = dHash/pHash distance, M = mirrored",
                 REV_DIR / f"{smp}_banda_{band}.png")
print(f"{len(REVIEW)} pares en {rev_csv}; láminas en {REV_DIR}")

### 16b. Resultado de la revisión: precisión por banda y elección de la regla

Con los veredictos se estiman:

* la precisión de cada banda (IC de Wilson al 95 %);
* el número real de imágenes de test con duplicado (estimador estratificado: tamaño de la banda × proporción de `same`);
* la precisión y la sensibilidad de varias reglas de decisión.

`uncertain` cuenta como no duplicado. La regla se elige sobre la misma muestra, así que sus cifras son optimistas.

In [ ]:
# Resultados de la revisión manual: precisión por banda (Wilson 95 %), estimación estratificada del número real de
# imágenes de test con duplicado en train/val y precisión/sensibilidad estimadas de varias reglas de decisión.
REVIEW = pd.read_csv(OUT / "revision_duplicados_muestra.csv", keep_default_na=False)
assert REVIEW.verdict.isin(["same", "different", "uncertain"]).all(), "Faltan veredictos en revision_duplicados_muestra.csv"
REVIEW["is_dup"] = REVIEW.verdict == "same"          # 'uncertain' cuenta como no duplicado (estimación conservadora)

band_rows = []
for kind, smp in (("d", "dHash"), ("p", "pHash")):
    dist = NN[("test", "train+val", kind)][0]
    for lo, hi in REVIEW_BANDS[kind]:
        b = f"{lo}" if lo == hi else f"{lo}-{hi}"
        g_ = REVIEW[(REVIEW["sample"] == smp) & (REVIEW.band == b)]
        k, n, N = int(g_.is_dup.sum()), len(g_), int(((dist >= lo) & (dist <= hi)).sum())
        lo_ci, hi_ci = wilson(k, n)
        band_rows.append(dict(hash=smp, band=b, test_images_in_band=N, reviewed=n, same=k, uncertain=int((g_.verdict == "uncertain").sum()),
                              precision=k / n, ci_low=lo_ci, ci_high=hi_ci, est_true_dups=N * k / n,
                              label_conflicts_among_same=int((g_.is_dup & (g_.label_test != g_.label_ref)).sum())))
DUP_REVIEW = pd.DataFrame(band_rows)
DUP_REVIEW.to_csv(OUT / "revision_duplicados_por_banda.csv", index=False)
display(DUP_REVIEW.round(3))

# peso de cada par revisado = tamaño de su banda / pares revisados en la banda (estimador estratificado)
REVIEW["w"] = [DUP_REVIEW.set_index(["hash", "band"]).loc[(r["sample"], r.band)].pipe(lambda x: x.test_images_in_band / x.reviewed)
               for _, r in REVIEW.iterrows()]
RULES = OrderedDict([
    ("dHash = 0 (Section 5)", lambda d, p: d <= 0),
    ("dHash <= 1", lambda d, p: d <= 1),
    ("dHash <= 4", lambda d, p: d <= 4),
    ("pHash <= 4", lambda d, p: p <= 4),
    ("pHash <= 6", lambda d, p: p <= 6),
    ("pHash <= 8", lambda d, p: p <= 8),
    ("dHash <= 8 and pHash <= 8", lambda d, p: (d <= 8) & (p <= 8)),
    ("dHash <= 10 and pHash <= 12", lambda d, p: (d <= 10) & (p <= 12)),
])
rule_rows = []
for name, f in RULES.items():
    row = dict(rule=name)
    for smp in ("dHash", "pHash"):
        r_ = REVIEW[REVIEW["sample"] == smp]; flag = f(r_.dhash_d.values, r_.phash_d.values); y_ = r_.is_dup.values; w = r_.w.values
        tp = (w * flag * y_).sum()
        row[f"precision_{smp}_sample"] = tp / (w * flag).sum() if flag.any() else float("nan")
        row[f"recall_{smp}_sample"] = tp / (w * y_).sum()
    rule_rows.append(row)
DUP_RULES = pd.DataFrame(rule_rows)
DUP_RULES.to_csv(OUT / "revision_duplicados_reglas.csv", index=False)
same = REVIEW[REVIEW.is_dup]
print(f"Pares confirmados: {len(same)}; con etiqueta distinta entre las dos copias: "
      f"{(same.label_test != same.label_ref).sum()} ({100 * (same.label_test != same.label_ref).mean():.1f} %); "
      f"especulares: {same.mirrored.sum()}")
for smp in ("dHash", "pHash"):
    print(f"Estimación (muestra {smp}) de imágenes de test con duplicado real en train/val: "
          f"{DUP_REVIEW[DUP_REVIEW.hash == smp].est_true_dups.sum():.0f} de {len(H['test'])}")
display(DUP_RULES.round(3))

# Rejilla de reglas conjuntas: imágenes de test marcadas con CUALQUIER pareja en train+val (no sólo el vecino más cercano)
# frente a los aciertos/fallos sobre los 360 pares revisados. Una regla útil marca un número de imágenes cercano a la
# estimación estratificada anterior y apenas tiene falsos positivos en la muestra.
q_, r_ = H["test"], H_trval
_D = [hamming_matrix(q_[c].values, r_.phash.values).astype(np.uint8) for c in ("phash", "dhash_flip")]
_P = [hamming_matrix(q_[c].values, r_.phash_dct.values).astype(np.uint8) for c in ("phash_dct", "phash_dct_flip")]
grid = []
for td in (0, 2, 4, 6, 8, 10, 64):
    for tp in (0, 4, 6, 8, 10, 12, 64):
        if td == tp == 64:
            continue
        hit = (((_D[0] <= td) & (_P[0] <= tp)) | ((_D[1] <= td) & (_P[1] <= tp))).any(1)
        f = (REVIEW.dhash_d <= td) & (REVIEW.phash_d <= tp)
        grid.append(dict(dhash_max=td, phash_max=tp, test_flagged=int(hit.sum()), reviewed_flagged=int(f.sum()),
                         reviewed_TP=int((f & REVIEW.is_dup).sum()), reviewed_FP=int((f & ~REVIEW.is_dup).sum()),
                         reviewed_FN=int((~f & REVIEW.is_dup).sum())))
del _D, _P
DUP_GRID = pd.DataFrame(grid).replace({64: "-"})
DUP_GRID.to_csv(OUT / "duplicados_rejilla_reglas.csv", index=False)
DUP_GRID

# Fig. 9: proporción de duplicados reales por banda de distancia (IC de Wilson) y ejemplos revisados
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharey=True)
for ax, smp, col in zip(axes, ("dHash", "pHash"), ("#2b6cb0", "#dd6b20")):
    t_ = DUP_REVIEW[DUP_REVIEW.hash == smp].copy()
    t_["lo_"] = t_.band.str.split("-").str[0].astype(int); t_ = t_.sort_values("lo_")
    x_ = np.arange(len(t_))
    ax.errorbar(x_, t_.precision, yerr=[t_.precision - t_.ci_low, t_.ci_high - t_.precision], fmt="o-", color=col, capsize=3)
    ax.set_xticks(x_); ax.set_xticklabels(t_.band, fontsize=8); ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel(f"{smp if smp == 'dHash' else 'pHash-DCT'} distance (bits)")
    for xi, n in zip(x_, t_.test_images_in_band):
        ax.annotate(f"{n}", (xi, 1.0), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=6, color="#718096")
axes[0].set_ylabel("Proportion of true duplicates\n(manual review, 20 pairs/band)")
fig.suptitle("Test image vs. nearest train/val image: proportion of true duplicates per distance band\n(grey numbers: test images in the band)", fontsize=9)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"fig9_revision_duplicados.{ext}", dpi=300, bbox_inches="tight")
plt.show(); plt.close(fig)

## 17. Reentrenamiento sin imágenes repetidas

La regla elegida en 16b (dHash ≤ 8 **y** pHash-DCT ≤ 8, en la misma orientación) se aplica para:

1. quitar de **train** toda imagen con copia en validación o test;
2. dejar una sola imagen por grupo de repetidas dentro de train, y quitar el grupo entero si sus etiquetas no coinciden;
3. quitar de **validación** las imágenes con copia en test (la parada temprana no debe ver el test).

El test **no se toca**: así los números son comparables con los del artículo. Además se informa del *test limpio*, sin
las imágenes que tienen copia en train/val según la misma regla. El balanceo se rehace desde el train depurado
(`train_balanced_dedup_*`). Se entrenan `completo` y `base` con las 5 semillas en `WORK_DIR/runs_dedup`.

In [ ]:
#@setup:dedup
from contextlib import contextmanager

# Regla elegida con la revisión manual de la sección 16 (rejilla de 16b): marca un número de imágenes de test
# (647) acorde con la estimación estratificada de duplicados reales, con 1 falso positivo en los 360 pares revisados
DEDUP_RULE = dict(dhash=8, phash=8)
DEDUP_VARIANTS, DEDUP_SEEDS = ["completo", "base"], CFG["ablation_seeds"]
RUNS_DEDUP = WORK / "runs_dedup"

def rule_pairs(hq, hr, same=False, chunk=256):
    """Pares (i, j) con dHash <= umbral y pHash-DCT <= umbral en la misma orientación (normal o especular)."""
    out = []
    for s in range(0, len(hq), chunk):
        e = slice(s, s + chunk); hit = None
        for cd, cp in (("phash", "phash_dct"), ("dhash_flip", "phash_dct_flip")):
            h_ = ((hamming_matrix(hq[cd].values[e], hr.phash.values) <= DEDUP_RULE["dhash"]) &
                  (hamming_matrix(hq[cp].values[e], hr.phash_dct.values) <= DEDUP_RULE["phash"]))
            hit = h_ if hit is None else hit | h_
        if same:
            ii = np.arange(hit.shape[0]); hit[ii, s + ii] = False
        ii, jj = np.nonzero(hit)
        out += list(zip(s + ii, jj))
    return out

def components(n, pairs):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in pairs:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return np.array([find(i) for i in range(n)])

t0 = time.time()
H_valtest = pd.concat([H["val"], H["test"]], ignore_index=True)
P_tr_vt = rule_pairs(H["train"], H_valtest)                 # train con copia en val o test
P_va_te = rule_pairs(H["val"], H["test"])                   # val con copia en test
P_te_tv = rule_pairs(H["test"], H_trval)                    # test con copia en train o val
P_tr_tr = rule_pairs(H["train"], H["train"], same=True)     # repetidas dentro de train
print(f"Pares según la regla en {time.time() - t0:.0f} s")

comp = components(len(H["train"]), P_tr_tr)
lab = H["train"].label.values
cdf = pd.DataFrame({"comp": comp, "label": lab})
n_lab = cdf.groupby("comp").label.nunique()
conflict_comps = set(n_lab[n_lab > 1].index)
first_of_comp = ~pd.Series(comp).duplicated().values
drop_train = set(i for i, _ in P_tr_vt)
drop_within = set(np.where(~first_of_comp)[0])
drop_conflict = set(np.where(np.isin(comp, list(conflict_comps)))[0])
keep_tr = np.array([i not in drop_train | drop_within | drop_conflict for i in range(len(H["train"]))])
drop_val = set(i for i, _ in P_va_te)
TEST_LEAK = set(H["test"].path.iloc[i] for i, _ in P_te_tv)

assert list(H["train"].path) == [p for p, _ in train_orig] and list(H["val"].path) == [p for p, _ in val_samples]
train_orig_dd = [s for s, k in zip(train_orig, keep_tr) if k]
val_dd = [s for i, s in enumerate(val_samples) if i not in drop_val]
train_gen_dd = build_balanced_train(train_orig_dd, CFG["seed"], tag="_dedup")
train_samples_dd = train_orig_dd + train_gen_dd
rng_dd = np.random.RandomState(CFG["seed"])
train_clean_dd = [train_orig_dd[i] for i in sorted(rng_dd.choice(len(train_orig_dd), min(CFG["clean_train_eval_n"], len(train_orig_dd)), replace=False))]
test_clean_mask = np.array([p not in TEST_LEAK for p, _ in test_samples])

DEDUP_SUMMARY = OrderedDict(
    train_original=len(train_orig), removed_dup_with_val_or_test=len(drop_train),
    removed_within_train_repeats=len(drop_within - drop_train),
    removed_label_conflict_clusters=len(drop_conflict - drop_within - drop_train),
    train_after=len(train_orig_dd), val_original=len(val_samples), val_removed_dup_with_test=len(drop_val),
    val_after=len(val_dd), test=len(test_samples), test_with_dup_in_train_or_val=len(TEST_LEAK),
    test_clean=int(test_clean_mask.sum()), train_generated_after=len(train_gen_dd))
json.dump(DEDUP_SUMMARY, open(OUT / "dedup_resumen.json", "w"), indent=2)
pd.DataFrame({"path": sorted(TEST_LEAK)}).to_csv(OUT / "test_con_duplicado_regla.csv", index=False)
pd.DataFrame({"Class": CLASSES, "Train original": counts(train_orig), "Train dedup": counts(train_orig_dd),
              "Val dedup": counts(val_dd), "Test": counts(test_samples),
              "Test with duplicate": counts([s for s in test_samples if s[0] in TEST_LEAK])}).to_csv(OUT / "dedup_por_clase.csv", index=False)

@contextmanager
def data_context(**kw):
    """Sustituye temporalmente los datos globales que usa train_run (train/val/test, runs)."""
    g_ = globals(); old = {k: g_[k] for k in kw}
    g_.update(kw)
    try:
        yield
    finally:
        g_.update(old)

DEDUP_CTX = dict(train_samples=train_samples_dd, val_samples=val_dd, train_clean_samples=train_clean_dd, RUNS_DIR=RUNS_DEDUP)
RUNS_DEDUP.mkdir(exist_ok=True)
pd.Series(DEDUP_SUMMARY)

In [ ]:
# Entrenamiento sin repetidas: 'completo' y 'base' x 5 semillas en WORK_DIR/runs_dedup (reanudable: reutiliza lo terminado).
# En el servidor se lanzaron en paralelo con worker_ext.py; aquí sólo se reutilizan.
RUN_DEDUP = True
if RUN_DEDUP:
    with data_context(**DEDUP_CTX):
        for v in DEDUP_VARIANTS:
            for s in DEDUP_SEEDS:
                train_run(v, s, verbose=False)

### 17b. Original frente a sin repetidas: test completo y test limpio

Se comparan media ± DT por semilla, la *t* pareada por semilla (original frente a sin repetidas) y el McNemar exacto
del conjunto promediado de semillas. La diferencia entre el test completo y el test limpio mide cuánto inflan la
exactitud las copias.

In [ ]:
def runs_preds(runs_dir, variant, seeds):
    out = {}
    for s in seeds:
        d = Path(runs_dir) / f"{variant}_s{s}"
        if (d / "meta.json").exists():
            out[s] = load_test_predictions(d)
    return out

def summarize(preds, mask, tta=True):
    accs, f1s, P = [], [], []
    for s, (y, p0, pt) in preds.items():
        p = (pt if tta else p0)
        a = p.argmax(1)
        accs.append((a[mask] == y[mask]).mean()); f1s.append(fast_macro_f1(y[mask], a[mask])); P.append(p)
    return np.array(accs), np.array(f1s), (np.mean(P, 0).argmax(1) if P else None)

DEDUP_ROWS, DEDUP_TESTS = [], []
subsets = {"full test": np.ones(len(test_samples), bool), "clean test": test_clean_mask}
for v in DEDUP_VARIANTS:
    orig, dd = runs_preds(RUNS_DIR, v, DEDUP_SEEDS), runs_preds(RUNS_DEDUP, v, DEDUP_SEEDS)
    common = sorted(set(orig) & set(dd))
    for sub, m in subsets.items():
        for name, pr in (("original", orig), ("deduplicated", dd)):
            a, f, _ = summarize(pr, m)
            DEDUP_ROWS.append(dict(variant=v, train_data=name, test=sub, n_seeds=len(a), n_test=int(m.sum()),
                                   acc_mean=a.mean() if len(a) else np.nan, acc_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                                   f1_mean=f.mean() if len(f) else np.nan, f1_sd=f.std(ddof=1) if len(f) > 1 else np.nan))
        if len(common) >= 2:
            ao, fo, eo = summarize({s: orig[s] for s in common}, m)
            ad, fd, ed = summarize({s: dd[s] for s in common}, m)
            y_ = y_test[m]
            DEDUP_TESTS.append(dict(variant=v, test=sub, seeds=len(common), d_acc=(ad - ao).mean(), d_f1=(fd - fo).mean(),
                                    t_p_acc=stats.ttest_rel(ad, ao).pvalue, t_p_f1=stats.ttest_rel(fd, fo).pvalue,
                                    mcnemar_ens_p=mcnemar_exact(ed[m] == y_, eo[m] == y_)["p_value"]))
DEDUP_RES = pd.DataFrame(DEDUP_ROWS); DEDUP_CMP = pd.DataFrame(DEDUP_TESTS)
DEDUP_RES.to_csv(OUT / "dedup_resultados.csv", index=False); DEDUP_CMP.to_csv(OUT / "dedup_contrastes.csv", index=False)

# completo frente a base después de depurar (¿se mantiene la conclusión de la ablación?)
cb = []
for sub, m in subsets.items():
    pc, pb = runs_preds(RUNS_DEDUP, "completo", DEDUP_SEEDS), runs_preds(RUNS_DEDUP, "base", DEDUP_SEEDS)
    common = sorted(set(pc) & set(pb))
    if len(common) >= 2:
        ac, fc, ec = summarize({s: pc[s] for s in common}, m); ab, fb, eb = summarize({s: pb[s] for s in common}, m)
        cb.append(dict(test=sub, seeds=len(common), d_acc_completo_minus_base=(ac - ab).mean(), t_p_acc=stats.ttest_rel(ac, ab).pvalue,
                       d_f1=(fc - fb).mean(), t_p_f1=stats.ttest_rel(fc, fb).pvalue,
                       mcnemar_ens_p=mcnemar_exact(ec[m] == y_test[m], eb[m] == y_test[m])["p_value"]))
DEDUP_CB = pd.DataFrame(cb); DEDUP_CB.to_csv(OUT / "dedup_completo_vs_base.csv", index=False)
display(DEDUP_RES.round(4)); display(DEDUP_CMP.round(4)); display(DEDUP_CB.round(4))

## 18. Segundo conjunto de datos: RAF-DB

Para comprobar si las conclusiones se generalizan, el mismo protocolo se repite **sin reajustar ningún hiperparámetro**
sobre RAF-DB (subconjunto básico, 7 emociones, 12 271 imágenes de entrenamiento y 3 068 de test, alineadas a 100x100):

* validación: 10 % estratificado del train oficial, separado antes de balancear;
* balanceo y aumento como en FER-2013; entrada en gris replicada a 3 canales, como en el resto del artículo;
* `completo` frente a `base`, 5 semillas, en `WORK_DIR/runs_rafdb`.

Además hay una **evaluación cruzada**: los modelos `completo` de FER-2013 se prueban en el test de RAF-DB y al revés.
La ruta se indica con `FER_RAFDB_DIR` (por defecto `~/TESIS/datasets_emociones/RAF-DB`). Si no existe, la sección se omite.

In [ ]:
#@setup:rafdb
from contextlib import contextmanager

# RAF-DB oficial (subconjunto básico): EmoLabel/list_patition_label.txt + Image/aligned/*_aligned.jpg (100x100).
# También acepta carpetas train/<clase>/ y test/<clase>/ con nombres de clase o con los códigos 1-7 de RAF-DB.
RAF_DIR = Path(os.environ.get("FER_RAFDB_DIR", "~/TESIS/datasets_emociones/RAF-DB"))
RAF_VARIANTS, RAF_SEEDS = ["completo", "base"], CFG["ablation_seeds"]
RUNS_RAF = WORK / "runs_rafdb"
RAF_CODE = {1: "surprise", 2: "fear", 3: "disgust", 4: "happy", 5: "sad", 6: "angry", 7: "neutral"}

def load_rafdb(root):
    lab = sorted(root.rglob("list_patition_label.txt"))
    if lab:
        img_dir = next((d for d in root.rglob("aligned") if d.is_dir()), None)
        assert img_dir is not None, f"No hay carpeta 'aligned' bajo {root}"
        split = {"train": [], "test": []}
        for line in open(lab[0]):
            name, code = line.split()
            p = img_dir / name.replace(".jpg", "_aligned.jpg")
            split["train" if name.startswith("train") else "test"].append((str(p), CLASSES.index(RAF_CODE[int(code)])))
        return split["train"], split["test"]
    def from_folders(d):
        rows = []
        for sub in sorted(Path(d).iterdir()):
            if sub.is_dir():
                c = RAF_CODE[int(sub.name)] if sub.name.isdigit() else sub.name.lower()
                c = {"anger": "angry", "happiness": "happy", "sadness": "sad", "surprised": "surprise"}.get(c, c)
                rows += [(str(f), CLASSES.index(c)) for f in sorted(sub.iterdir()) if f.suffix.lower() in IMG_EXT]
        return rows
    tr = next(d for d in root.rglob("train") if d.is_dir()); te = next(d for d in root.rglob("test") if d.is_dir())
    return from_folders(tr), from_folders(te)

if not 'data_context' in globals():
    @contextmanager
    def data_context(**kw):
        """Sustituye temporalmente los datos globales que usa train_run (train/val/test, runs)."""
        g_ = globals(); old = {k: g_[k] for k in kw}
        g_.update(kw)
        try:
            yield
        finally:
            g_.update(old)

RAF_OK = RAF_DIR.exists()
if RAF_OK:
    raf_train_all, raf_test = load_rafdb(RAF_DIR)
    missing = [p for p, _ in raf_train_all + raf_test if not Path(p).exists()]
    assert not missing, f"{len(missing)} imágenes de la lista no existen, p. ej. {missing[:2]}"
    # validación: 10 % estratificado de train ANTES de balancear (RAF-DB no trae validación), misma semilla
    tr_i, va_i = train_test_split(np.arange(len(raf_train_all)), test_size=CFG["val_fraction"], random_state=CFG["seed"],
                                  stratify=[l for _, l in raf_train_all])
    raf_train_orig = [raf_train_all[i] for i in sorted(tr_i)]; raf_val = [raf_train_all[i] for i in sorted(va_i)]
    raf_gen = build_balanced_train(raf_train_orig, CFG["seed"], tag="_rafdb")
    raf_train = raf_train_orig + raf_gen
    rng_r = np.random.RandomState(CFG["seed"])
    raf_clean = [raf_train_orig[i] for i in sorted(rng_r.choice(len(raf_train_orig), min(CFG["clean_train_eval_n"], len(raf_train_orig)), replace=False))]
    RAF_CTX = dict(train_samples=raf_train, val_samples=raf_val, test_samples=raf_test, train_clean_samples=raf_clean, RUNS_DIR=RUNS_RAF)
    RUNS_RAF.mkdir(exist_ok=True)
    RAF_TABLE = pd.DataFrame({"Class": CLASSES, "Train original": counts(raf_train_orig), "Train generated": counts(raf_gen),
                              "Validation": counts(raf_val), "Test": counts(raf_test)})
    RAF_TABLE.to_csv(OUT / "rafdb_datos.csv", index=False)
    display(RAF_TABLE)
else:
    print(f"RAF-DB no encontrado en {RAF_DIR}: define FER_RAFDB_DIR o copia ahí el subconjunto básico. Se omite la sección 18.")

In [ ]:
# Entrenamiento en RAF-DB con exactamente el mismo protocolo e hiperparámetros que en FER-2013 (sin reajustar nada).
# En el servidor se lanzó en paralelo con worker_ext.py; aquí sólo se reutiliza lo terminado.
if RAF_OK:
    with data_context(**RAF_CTX):
        for v in RAF_VARIANTS:
            for s in RAF_SEEDS:
                train_run(v, s, verbose=False)

In [ ]:
# RAF-DB: completo frente a base (media ± DT, t pareada, McNemar del conjunto) y evaluación cruzada entre conjuntos
# (modelos de FER-2013 sobre el test de RAF-DB y al revés), con la misma TTA de 3 vistas.
@torch.no_grad()
def cross_eval(run_dir, spec, samples):
    m = ResNet50CBAM(NUM_CLASSES, tuple(spec["cbam"]), pretrained=False).to(device)
    m.load_state_dict(torch.load(Path(run_dir) / "best.pth", map_location=device, weights_only=True))
    _, pt, _ = predict(m, make_loader(samples, None, False, tta=True), tta=True)
    del m; torch.cuda.empty_cache()
    return pt

RAF_RES = RAF_CMP = RAF_CROSS = None
test_samples_fer, RUNS_FER = test_samples, RUNS_DIR
if RAF_OK:
    with data_context(**RAF_CTX):
        y_raf = labels_of(raf_test)
        rp = {v: runs_preds(RUNS_RAF, v, RAF_SEEDS) for v in RAF_VARIANTS}
        rows = []
        for v in RAF_VARIANTS:
            for tta in (False, True):
                a, f, _ = summarize(rp[v], np.ones(len(raf_test), bool), tta)
                rows.append(dict(variant=v, tta=tta, n_seeds=len(a), acc_mean=a.mean(), acc_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                                 f1_mean=f.mean(), f1_sd=f.std(ddof=1) if len(f) > 1 else np.nan))
        RAF_RES = pd.DataFrame(rows)
        common = sorted(set(rp["completo"]) & set(rp["base"]))
        if len(common) >= 2:
            all_ = np.ones(len(raf_test), bool)
            ac, fc, ec = summarize({s: rp["completo"][s] for s in common}, all_)
            ab, fb, eb = summarize({s: rp["base"][s] for s in common}, all_)
            RAF_CMP = dict(seeds=len(common), d_acc=(ac - ab).mean(), t_p_acc=stats.ttest_rel(ac, ab).pvalue, d_f1=(fc - fb).mean(),
                           t_p_f1=stats.ttest_rel(fc, fb).pvalue, mcnemar_ens_p=mcnemar_exact(ec == y_raf, eb == y_raf)["p_value"],
                           ens_acc_completo=(ec == y_raf).mean(), ens_f1_completo=fast_macro_f1(y_raf, ec))
            RAF_PC = per_class_table(y_raf, ec); RAF_PC.to_csv(OUT / "rafdb_por_clase_ensemble_completo.csv", index=False)
            display(RAF_PC.round(3))
        # evaluación cruzada: sólo 'completo', semillas comunes
        cross = []
        for s in RAF_SEEDS:
            fer_run, raf_run = RUNS_FER / f"completo_s{s}", RUNS_RAF / f"completo_s{s}"
            if (fer_run / "meta.json").exists():
                pt = cross_eval(fer_run, VARIANTS["completo"], raf_test)
                cross.append(dict(train="FER-2013", test="RAF-DB", seed=s, acc=(pt.argmax(1) == y_raf).mean(), f1=fast_macro_f1(y_raf, pt.argmax(1))))
            if (raf_run / "meta.json").exists():
                pt = cross_eval(raf_run, VARIANTS["completo"], test_samples_fer)
                cross.append(dict(train="RAF-DB", test="FER-2013", seed=s, acc=(pt.argmax(1) == y_test).mean(), f1=fast_macro_f1(y_test, pt.argmax(1))))
        RAF_CROSS = pd.DataFrame(cross)
    if len(RAF_CROSS):
        RAF_CROSS.to_csv(OUT / "cruzado_fer_rafdb_por_semilla.csv", index=False)
        display(RAF_CROSS.groupby(["train", "test"])[["acc", "f1"]].agg(["mean", "std"]).round(4))
    RAF_RES.to_csv(OUT / "rafdb_resultados.csv", index=False)
    json.dump(RAF_CMP, open(OUT / "rafdb_completo_vs_base.json", "w"), indent=2, default=float)
    display(RAF_RES.round(4)); print(RAF_CMP)

# Auditoría de duplicados en RAF-DB con la misma regla de la sección 17 (test frente a train+val)
RAF_DUP = None
if RAF_OK:
    HR = {k: extra_hashes(hash_table(sm, f"rafdb_{k}"), f"rafdb_{k}") for k, sm in (("trainval", raf_train_orig + raf_val), ("test", raf_test))}
    raf_leak = sorted(set(i for i, _ in rule_pairs(HR["test"], HR["trainval"])))
    RAF_DUP = dict(test_with_dup_in_trainval=len(raf_leak), pct=100 * len(raf_leak) / len(raf_test),
                   md5_exact=int(HR["test"].md5.isin(set(HR["trainval"].md5)).sum()))
    json.dump(RAF_DUP, open(OUT / "rafdb_duplicados.json", "w"), indent=2)
    print("RAF-DB, duplicados test-train/val:", RAF_DUP)